# Day22A — MJ1 low-amplitude full-protocol audit  
## 0.3C DC vs 0.3C + 0.4C DC–AC

## Purpose

This notebook applies the Day21A full-protocol segmentation audit pipeline to a lower-amplitude MJ1 experimental group:

- `0.3C DC`
- `0.3C + 0.4C DC–AC`

The objective is to test whether the boundary/control-state mediated first-passage gain observed in the `0.3C + 0.7C` group remains above audit resolution, weakens toward the resolution floor, or becomes unresolved when the AC amplitude is reduced.

This notebook is not a new mechanism search. It is a controlled extension of the Day21A audit framework to one low-amplitude comparison group.

---

## Relation to Day21A

Day22A inherits the Day21A audit contract unless explicitly stated otherwise.

The following definitions and rules remain unchanged:

- strict-net signed Q integration
- no rectification
- no cumulative maximum
- first-passage time at equal `Q_net`
- Q80/Q90 evaluated using both nominal and common-capacity anchors
- Segment A/B/D segmentation
- Segment-A residual defined only in the shared prescribed-current region
- prescribed-geometry residual retained as the formal audit quantity
- fitted-waveform geometry used only as diagnostic interpretation
- formal verdict kept separate from diagnostic interpretation
- raw `Δt(Q)` treated as real but not mechanism-pure

The Day21A closure reference is:

`docs/day21A_close.md`

The Day22A planning reference is:

`docs/day22A_plan_low_amplitude_mj1_audit.md`

---

## Experimental resolution floor and verdict limitation

Day22A is a low-amplitude audit. Therefore, the interpretation depends critically on the experimental resolution floor.

The PyBaMM Day18B numerical-null floor is not used as an experimental noise floor. That floor belongs to the simulation / solver domain and cannot be transferred directly to the MJ1 experimental domain.

For MJ1 experimental data, the relevant uncertainty sources include:

- NGU201 1 Hz logging quantization
- finite current and voltage measurement precision
- timestamp jitter or reconstructed time-axis uncertainty
- current-waveform realization error relative to the prescribed signal
- thermal drift and OCV drift during the protocol
- lack of repeat experiments for the same protocol

Because no independent repeat-based experimental noise floor is available, Day22A must not claim strict disappearance of an effect.

Instead, Day22A uses the following interpretation language:

- `above audit resolution`: the signal is large enough to be interpreted relative to the Day22A audit floor.
- `below audit resolution`: the signal is not distinguishable from the available resolution limits.
- `unresolved`: the signal cannot be classified because the relevant floor estimate is unavailable or contaminated.

The term `disappears` is not used as a formal verdict unless an independent repeat-based experimental noise floor is established.

### Required resolution diagnostics

Before any mechanism verdict, Day22A must include a resolution-floor diagnostic layer.

At minimum, the notebook must report one of the following:

1. A DC self-consistency lower-bound floor, obtained by splitting the `0.3C DC` reference trajectory into two internally consistent subsets and computing a self-`Δt(Q)` diagnostic. This estimates a lower bound for integration, interpolation, and first-passage numerical resolution. It does not estimate between-run repeatability.

2. A waveform-fidelity diagnostic, obtained by comparing measured current against the prescribed current geometry in Segment A and reporting current RMSD, accumulated Q error, and first-passage error.

3. If neither diagnostic is available, the notebook must explicitly report:

`no_independent_experimental_noise_floor_available`

and all low-amplitude claims must be limited to:

`effect below current audit resolution`

rather than:

`effect disappears`.

### Day22A verdict language

Day22A may compare the low-amplitude `0.3C + 0.4C` group against the Day21A `0.3C + 0.7C` group only in bounded terms.

Allowed:

- the first-passage gain is smaller than in Day21A;
- the gain remains above the Day22A audit-resolution floor;
- the gain is below the available audit resolution;
- the result is unresolved because the signal magnitude approaches the experimental floor.

Not allowed:

- the effect disappears, unless repeat-based noise-floor evidence is available;
- a small residual proves persistence of the mechanism;
- PyBaMM numerical-null floor is used as the MJ1 experimental floor;
- fitted-waveform residual is used as a formal replacement for the prescribed-geometry residual.

---

## Day22A group definition

The intended Day22A group is:

- DC reference: `0.3C DC`
- DC–AC protocols: `0.3C + 0.4C` at available frequency labels

The preferred frequency-label set is:

- `0.1τ`
- `1τ`
- `10τ`

The corresponding raw-file directory is:

`data/raw_mj1_ngu201_day22A_0p3C_0p4C/`

Expected file names:

- `MJ1_0p3C_DC_NGU201_raw.csv`
- `MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv`
- `MJ1_0p3C_0p4C_1tau_NGU201_raw.csv`
- `MJ1_0p3C_0p4C_10tau_NGU201_raw.csv`

---

## Known data-format issue

Unlike Day21A, some Day22A files may not be pure NGU201 LOG exports.

Two CSV formats must be supported:

### Type A — NGU201 LOG raw format

Contains NGU201 metadata rows, for example:

- `#Device,NGU201`
- `#Logging Interval[s],1`
- `#Start Time,...`

with a data header such as:

`Timestamp,U1[V],I1[A],P1[W],DVM1[V]`

### Type B — processed 1 Hz aligned format

A previously processed CSV in which the time column is already aligned to the 1 Hz NGU201 sampling interval, typically increasing by 1 s per row.

For Type B files, the time series may use numeric or reconstructed monotonic seconds rather than the original NGU201 timestamp strings.

This notebook must therefore explicitly audit:

- `csv_format`
- `time_column_name`
- `time_parse_method`
- `time_reconstructed_from_row_index`
- `time_monotonic_status`

No silent time-axis repair is allowed.

If a time column is non-monotonic or wraps after one hour, the loader must either unwrap it explicitly or reconstruct `t_s` from row index at 1 Hz, and this must be recorded in the inventory.

---

## Temperature metadata

The experiment was conducted without a temperature chamber.

Temperature is handled as protocol-level summary metadata, not as a sample-by-sample signal aligned to NGU201 voltage/current.

Expected temperature fields:

- `T_surface_max_C`
- `T_surface_mean_C`

These values originate from Pt100 cell-surface temperature measurement, with the sensor attached axially to the cell surface.

---

## Day22A audit sequence

The notebook will proceed in staged form:

1. **Cell 0A — raw CSV format inspection**  
   Inspect raw files and classify CSV format before any trajectory processing.

2. **Cell 1 — audit constants and schema**  
   Reuse Day21A constants and extend inventory schema only where needed for CSV format and time-axis provenance.

3. **Cell 2 — file inventory**  
   Build Day22A file-level provenance inventory.

4. **Cell 3 — trajectory loading and sanity checks**  
   Load trajectories, standardize time axis, and trim charge-onset region.

5. **Cell 4 — event / AC-off audit**  
   Detect Vmax and AC-off timing.

6. **Cell 5 — strict-net Q integration and final-Q consistency**  
   Compute `Q_net(t)` and final-Q consistency.

7. **Cell 5A — experimental resolution-floor diagnostic**  
   Estimate a Day22A audit-resolution scale using DC self-consistency and/or waveform-fidelity diagnostics before interpreting low-amplitude residuals.

8. **Cell 6 — event-charge extraction and segment assignment**  
   Assign Q80/Q90 nominal/common anchors to Segment A/B/D/outside.

9. **Cell 7 — Δt(Q), Segment-A residual, and diagnostics**  
   Compute raw, prescribed-geometry, and diagnostic fitted-waveform quantities.

10. **Cell 8 — Day22A verdict**  
    Apply the same formal verdict discipline as Day21A, with low-amplitude audit-resolution caveats.

11. **Cell 9 — closure note**  
    Close the Day22A audit without expanding to additional groups.

---

## Boundaries

This notebook must not:

- redefine Segment-A residual thresholds
- treat fitted-waveform residual as a formal replacement for prescribed-geometry residual
- use the PyBaMM numerical-null floor as the MJ1 experimental floor
- claim effect disappearance without independent repeat-based noise-floor evidence
- search for new electrochemical mechanisms
- expand to all MJ1 groups at once
- modify Day21A outputs
- commit raw CSV files to Git

The first operational step is only CSV-format inspection.

In [2]:
# Day22A Cell 0 — minimal setup
# Defines repo/data paths and shared constants needed before format inspection.

from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Optional, Sequence
import json
import subprocess
import csv
import re

import numpy as np
import pandas as pd

REPO = Path("/Users/louislu/pybamm-dcac-superimposed").expanduser()
DATA_DIR = REPO / "data"
NOTEBOOK_NAME = "26_day22A_MJ1_low_amplitude_0p3C_0p4C_audit.ipynb"

RAW_DIR_DAY22A = DATA_DIR / "raw_mj1_ngu201_day22A_0p3C_0p4C"
METADATA_DIR = DATA_DIR / "metadata"

UNKNOWN = "unknown_not_recorded"

# Core inherited constants from Day21A
SOURCE_TYPE_MJ1 = "experimental_MJ1"
CELL_ID = "LG_INR18650_MJ1"
CHEMISTRY_FAMILY = "NMC_layered_oxide"

Q_NOM_AH = 3.4
ONE_C_A = 3.4
VMAX_V = 4.2

I_CUTOFF_A = 0.05
I_CUTOFF_DEFINITION = "absolute_50mA_first_reached"

I_CHARGE_ONSET_THRESHOLD_A = 0.05
I_CHARGE_ONSET_MIN_CONSECUTIVE_SAMPLES = 3

PHASE_CONVENTION = "charge_first"

AMBIENT_TEMPERATURE_C = 20.0
TEMPERATURE_CONTROL_TYPE = "ambient_lab_no_chamber"
TEMPERATURE_SENSOR_TYPE = "Pt100"
TEMPERATURE_SENSOR_PLACEMENT = "axial_cell_surface"
TEMPERATURE_DATA_SOURCE = "measured_surface_temperature"
TEMPERATURE_ALIGNMENT_METHOD = "segment_level_summary_only"
TEMPERATURE_TO_NGU201_ALIGNMENT_REQUIRED = False

DEFAULT_VOLTAGE_SOURCE = "NGU201"
DEFAULT_CURRENT_SOURCE = "NGU201"
DEFAULT_TIMEBASE_SOURCE = "NGU201_single_timebase_or_processed_1Hz"
DEFAULT_TIME_ALIGNMENT_METHOD = "native_or_processed_monotonic_timebase"
DEFAULT_VOLTAGE_CURRENT_ALIGNMENT_STATUS = "aligned_same_record"

TAU_LABEL_S = 11.1

def compute_tau_eff_s(tau_label_s: float, m_tau: float) -> float:
    if not np.isfinite(tau_label_s) or not np.isfinite(m_tau):
        return np.nan
    return float(m_tau * tau_label_s)

def compute_frequency_hz_from_tau_eff(tau_eff_s: float) -> float:
    if not np.isfinite(tau_eff_s) or tau_eff_s <= 0:
        return np.nan
    return float(1.0 / (2.0 * np.pi * tau_eff_s))

def compute_t_ac_s(frequency_hz: float) -> float:
    if not np.isfinite(frequency_hz) or frequency_hz <= 0:
        return np.nan
    return float(1.0 / frequency_hz)

print("[OK] Day22A minimal setup loaded.")
print(f"[OK] REPO = {REPO}")
print(f"[OK] DATA_DIR = {DATA_DIR}")
print(f"[OK] RAW_DIR_DAY22A = {RAW_DIR_DAY22A}")

[OK] Day22A minimal setup loaded.
[OK] REPO = /Users/louislu/pybamm-dcac-superimposed
[OK] DATA_DIR = /Users/louislu/pybamm-dcac-superimposed/data
[OK] RAW_DIR_DAY22A = /Users/louislu/pybamm-dcac-superimposed/data/raw_mj1_ngu201_day22A_0p3C_0p4C


In [3]:
# Day22A Cell 0A — raw CSV format inspection
#
# Purpose:
# - Inspect Day22A raw CSV files before any trajectory processing
# - Identify NGU201 raw LOG format vs processed 1 Hz aligned format
# - Inspect header lines, delimiter, candidate columns, and time-column style
#
# Explicitly NOT done here:
# - No trajectory loading
# - No Q integration
# - No Vmax / AC-off detection
# - No Δt calculation
# - No verdict

from pathlib import Path
import csv
import re
import numpy as np
import pandas as pd

RAW_DIR_DAY22A = DATA_DIR / "raw_mj1_ngu201_day22A_0p3C_0p4C"

EXPECTED_DAY22A_RAW_FILES = [
    "MJ1_0p3C_DC_NGU201_raw.csv",
    "MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv",
    "MJ1_0p3C_0p4C_1tau_NGU201_raw.csv",
    "MJ1_0p3C_0p4C_10tau_NGU201_raw.csv",
]

if not RAW_DIR_DAY22A.exists():
    raise FileNotFoundError(f"Day22A raw directory not found: {RAW_DIR_DAY22A}")

csv_paths = sorted(RAW_DIR_DAY22A.glob("*.csv"))
found_names = [p.name for p in csv_paths]

missing_expected = [name for name in EXPECTED_DAY22A_RAW_FILES if name not in found_names]
unexpected_files = [name for name in found_names if name not in EXPECTED_DAY22A_RAW_FILES]

print(f"[scan] RAW_DIR_DAY22A = {RAW_DIR_DAY22A}")
print(f"[scan] found CSV files = {len(csv_paths)}")
print(f"[scan] expected files missing = {missing_expected}")
print(f"[scan] unexpected CSV files = {unexpected_files}")

if missing_expected:
    raise FileNotFoundError(
        "Missing expected Day22A raw files:\n" + "\n".join(missing_expected)
    )


# -----------------------------------------------------------------------------
# 0A.1 Helpers
# -----------------------------------------------------------------------------

def sniff_delimiter_day22(path: Path, sample_bytes: int = 8192) -> str:
    """Best-effort delimiter detection."""
    try:
        sample = path.read_text(errors="ignore")[:sample_bytes]
        dialect = csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"])
        return dialect.delimiter
    except Exception:
        return ","


def split_line_day22(line: str, delimiter: str) -> list[str]:
    return [p.strip() for p in line.rstrip("\n").split(delimiter)]


def looks_like_ngu201_log_header(parts: list[str]) -> bool:
    lowered = [p.lower().strip() for p in parts]
    return (
        "timestamp" in lowered
        and "u1[v]" in lowered
        and "i1[a]" in lowered
    )


def looks_like_processed_header(parts: list[str]) -> bool:
    """
    Looser check for processed CSV tables.
    Accepts time/t_s/timestamp + voltage/current-like columns.
    """
    lowered = [p.lower().strip() for p in parts]

    has_time = any(
        c in {"time", "t", "t_s", "time_s", "timestamp", "time[s]", "t[s]"}
        or "time" in c
        for c in lowered
    )

    has_voltage = any(
        c in {"u", "u_v", "u1[v]", "voltage", "voltage_v", "u[v]", "v"}
        or "volt" in c
        or "[v]" in c
        for c in lowered
    )

    has_current = any(
        c in {"i", "i_a", "i1[a]", "current", "current_a", "i[a]"}
        or "curr" in c
        or "[a]" in c
        for c in lowered
    )

    return has_time and has_voltage and has_current


def detect_header_line_day22(path: Path, delimiter: str, max_scan_lines: int = 200) -> dict:
    """
    Detect whether file is:
    - NGU201_LOG_raw
    - processed_1Hz_aligned_candidate
    - unknown_requires_manual_review
    """
    lines = path.read_text(errors="ignore").splitlines()

    logging_interval_s = np.nan
    source_date = UNKNOWN
    start_time = UNKNOWN

    for i, line in enumerate(lines[:max_scan_lines]):
        parts = split_line_day22(line, delimiter)
        if not parts:
            continue

        key = parts[0].strip()

        if key == "#Logging Interval[s]" and len(parts) >= 2:
            try:
                logging_interval_s = float(parts[1])
            except Exception:
                logging_interval_s = np.nan

        if key == "#Date" and len(parts) >= 2:
            source_date = str(parts[1]).strip() if str(parts[1]).strip() else UNKNOWN

        if key == "#Start Time" and len(parts) >= 2:
            start_time = str(parts[1]).strip() if str(parts[1]).strip() else UNKNOWN

        if looks_like_ngu201_log_header(parts):
            return {
                "csv_format": "NGU201_LOG_raw",
                "header_line_idx": i,
                "columns": parts,
                "logging_interval_s": logging_interval_s,
                "source_date": source_date,
                "start_time": start_time,
            }

    # If not NGU201 LOG, inspect first plausible processed header.
    for i, line in enumerate(lines[:max_scan_lines]):
        parts = split_line_day22(line, delimiter)
        if len(parts) > 1 and looks_like_processed_header(parts):
            return {
                "csv_format": "processed_1Hz_aligned_candidate",
                "header_line_idx": i,
                "columns": parts,
                "logging_interval_s": logging_interval_s,
                "source_date": source_date,
                "start_time": start_time,
            }

    return {
        "csv_format": "unknown_requires_manual_review",
        "header_line_idx": None,
        "columns": [],
        "logging_interval_s": logging_interval_s,
        "source_date": source_date,
        "start_time": start_time,
    }


def inspect_time_column_style(path: Path, delimiter: str, header_line_idx: int | None, columns: list[str]) -> dict:
    """
    Inspect first non-NaN values in the likely time column.
    This does not load the full trajectory for analysis.
    """
    if header_line_idx is None or not columns:
        return {
            "time_column_name": UNKNOWN,
            "time_value_examples": [],
            "time_column_style": "unknown_no_header",
            "time_numeric_monotonic_first_20": UNKNOWN,
        }

    try:
        df_head = pd.read_csv(
            path,
            sep=delimiter,
            skiprows=header_line_idx,
            nrows=80,
            engine="python",
        )
    except Exception as exc:
        return {
            "time_column_name": UNKNOWN,
            "time_value_examples": [],
            "time_column_style": f"unreadable:{type(exc).__name__}",
            "time_numeric_monotonic_first_20": UNKNOWN,
        }

    candidate_time_cols = []
    for c in df_head.columns:
        cl = str(c).lower().strip()
        if cl in {"timestamp", "time", "t", "t_s", "time_s", "time[s]", "t[s]"} or "time" in cl:
            candidate_time_cols.append(c)

    if not candidate_time_cols:
        return {
            "time_column_name": UNKNOWN,
            "time_value_examples": [],
            "time_column_style": "unknown_no_time_column",
            "time_numeric_monotonic_first_20": UNKNOWN,
        }

    time_col = candidate_time_cols[0]
    vals = df_head[time_col].dropna().astype(str).head(20).tolist()

    numeric_vals = pd.to_numeric(pd.Series(vals), errors="coerce")
    n_numeric = int(numeric_vals.notna().sum())

    if n_numeric >= max(3, len(vals) // 2):
        arr = numeric_vals.dropna().to_numpy(dtype=float)
        monotonic = bool(np.all(np.diff(arr) >= 0)) if len(arr) >= 2 else UNKNOWN
        style = "numeric_seconds_or_index_like"
    else:
        monotonic = UNKNOWN
        if any(v.count(":") >= 1 for v in vals):
            style = "timestamp_string_colon_format"
        else:
            style = "non_numeric_string_time"

    return {
        "time_column_name": str(time_col),
        "time_value_examples": vals[:8],
        "time_column_style": style,
        "time_numeric_monotonic_first_20": monotonic,
    }


# -----------------------------------------------------------------------------
# 0A.2 Inspect files
# -----------------------------------------------------------------------------

format_rows = []

for path in csv_paths:
    delimiter = sniff_delimiter_day22(path)
    header_info = detect_header_line_day22(path, delimiter)
    time_info = inspect_time_column_style(
        path=path,
        delimiter=delimiter,
        header_line_idx=header_info["header_line_idx"],
        columns=header_info["columns"],
    )

    row = {
        "file_name": path.name,
        "file_size_kb": path.stat().st_size / 1024,
        "delimiter": repr(delimiter),
        **header_info,
        **time_info,
    }
    format_rows.append(row)

    print("\n" + "=" * 120)
    print(f"[FILE] {path.name}")
    print("=" * 120)
    print(f"csv_format: {header_info['csv_format']}")
    print(f"header_line_idx: {header_info['header_line_idx']}")
    print(f"columns: {header_info['columns']}")
    print(f"logging_interval_s: {header_info['logging_interval_s']}")
    print(f"source_date: {header_info['source_date']}")
    print(f"start_time: {header_info['start_time']}")
    print(f"time_column_name: {time_info['time_column_name']}")
    print(f"time_column_style: {time_info['time_column_style']}")
    print(f"time_numeric_monotonic_first_20: {time_info['time_numeric_monotonic_first_20']}")
    print(f"time_value_examples: {time_info['time_value_examples']}")

    # Print first 30 raw lines for manual audit.
    lines = path.read_text(errors="ignore").splitlines()
    print("\n[first 30 lines]")
    for i, line in enumerate(lines[:30]):
        print(f"{i:03d}: {line}")

df_day22A_format_inventory = pd.DataFrame(format_rows)

OUT_DAY22A_FORMAT_INVENTORY = DATA_DIR / "day22A_step0A_raw_csv_format_inventory.csv"
df_day22A_format_inventory.to_csv(OUT_DAY22A_FORMAT_INVENTORY, index=False)

print("\n" + "=" * 120)
print("[SUMMARY]")
print("=" * 120)
print(df_day22A_format_inventory[[
    "file_name",
    "csv_format",
    "header_line_idx",
    "time_column_name",
    "time_column_style",
    "time_numeric_monotonic_first_20",
    "logging_interval_s",
]].to_string(index=False))

print(f"\n[OK] Wrote Day22A raw CSV format inventory: {OUT_DAY22A_FORMAT_INVENTORY}")
print("[OK] Cell 0A completed. No trajectory processing was performed.")

[scan] RAW_DIR_DAY22A = /Users/louislu/pybamm-dcac-superimposed/data/raw_mj1_ngu201_day22A_0p3C_0p4C
[scan] found CSV files = 4
[scan] expected files missing = []
[scan] unexpected CSV files = []

[FILE] MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv
csv_format: NGU201_LOG_raw
header_line_idx: 0
columns: ['Timestamp', 'U1[V]', 'I1[A]', '']
logging_interval_s: nan
source_date: unknown_not_recorded
start_time: unknown_not_recorded
time_column_name: Timestamp
time_column_style: timestamp_string_colon_format
time_numeric_monotonic_first_20: unknown_not_recorded
time_value_examples: ['0:00:01', '0:00:02', '0:00:03', '0:00:04', '0:00:05', '0:00:06', '0:00:07', '0:00:08']

[first 30 lines]
000: Timestamp,U1[V],I1[A],
001: 0:00:01,2.953340E+00,1.718790E+00,
002: 0:00:02,3.019064E+00,2.368131E+00,
003: 0:00:03,3.013130E+00,1.992451E+00,
004: 0:00:04,2.951190E+00,8.773400E-01,
005: 0:00:05,2.875288E+00,-1.29E-01,
006: 0:00:06,2.845303E+00,-2.62E-01,
007: 0:00:07,2.901816E+00,5.791698E-01,
008: 0:00:08,2.99

In [4]:
# Day22A Cell 0B — full timebase and processed-format audit
#
# Purpose:
# - Reclassify header-line-0 files without NGU201 metadata as processed_1Hz_aligned_no_metadata
# - Parse full Timestamp column
# - Check monotonicity, rollover, and sampling interval
#
# Explicitly NOT done here:
# - No charge-onset trimming
# - No Q integration
# - No event detection
# - No verdict

OUT_DAY22A_TIMEBASE_AUDIT = DATA_DIR / "day22A_step0B_timebase_audit.csv"

def parse_day22_timestamp_to_seconds(ts: object) -> float:
    """
    Parse Day22A timestamp strings.

    Supported:
    - '0:00:01'       -> HH:MM:SS
    - '10:40:17.3'    -> HH:MM:SS.s
    - '44:26.0'       -> MM:SS.s
    - numeric seconds
    """
    if pd.isna(ts):
        return np.nan

    s = str(ts).strip()
    if not s:
        return np.nan

    parts = s.split(":")
    try:
        if len(parts) == 3:
            h = float(parts[0])
            m = float(parts[1])
            sec = float(parts[2])
            return h * 3600.0 + m * 60.0 + sec

        if len(parts) == 2:
            m = float(parts[0])
            sec = float(parts[1])
            return m * 60.0 + sec

        return float(s)
    except Exception:
        return np.nan


def unwrap_time_if_needed(raw_seconds: np.ndarray, raw_strings: Sequence[object]) -> tuple[np.ndarray, int, float]:
    """
    Monotonic unwrap for timestamp rollover.

    Returns:
    - unwrapped time seconds
    - number of unwrap events
    - rollover period used
    """
    t = np.asarray(raw_seconds, dtype=float).copy()

    colon_counts = []
    for x in raw_strings:
        if pd.isna(x):
            colon_counts.append(0)
        else:
            colon_counts.append(str(x).count(":"))

    # HH:MM:SS uses 24 h rollover; MM:SS uses 1 h rollover.
    rollover_period_s = 86400.0 if max(colon_counts, default=0) >= 2 else 3600.0

    offset = 0.0
    prev = np.nan
    n_unwrap = 0

    for i, val in enumerate(t):
        if not np.isfinite(val):
            t[i] = np.nan
            continue

        candidate = val + offset
        if np.isfinite(prev) and candidate < prev - 1e-6:
            offset += rollover_period_s
            n_unwrap += 1
            candidate = val + offset

        t[i] = candidate
        prev = candidate

    return t, n_unwrap, rollover_period_s


def inspect_day22_file_timebase(path: Path) -> dict[str, object]:
    delimiter = sniff_delimiter_day22(path)
    header_info = detect_header_line_day22(path, delimiter)

    header_line_idx = header_info["header_line_idx"]
    if header_line_idx is None:
        return {
            "file_name": path.name,
            "csv_format_refined": "unknown_requires_manual_review",
            "header_line_idx": np.nan,
            "time_column_name": UNKNOWN,
            "n_rows": 0,
            "time_parse_method": "unresolved_no_header",
            "time_monotonic_status": "unresolved",
            "time_unwrap_count": np.nan,
            "time_reconstructed_from_row_index": False,
            "dt_median_s": np.nan,
            "dt_min_s": np.nan,
            "dt_max_s": np.nan,
            "t_start_s": np.nan,
            "t_end_s": np.nan,
            "duration_s": np.nan,
            "notes": "no header detected",
        }

    df = pd.read_csv(
        path,
        sep=delimiter,
        skiprows=int(header_line_idx),
        engine="python",
    )

    # Identify time column
    time_cols = []
    for c in df.columns:
        cl = str(c).lower().strip()
        if cl in {"timestamp", "time", "t", "t_s", "time_s", "time[s]", "t[s]"} or "time" in cl:
            time_cols.append(c)

    if not time_cols:
        return {
            "file_name": path.name,
            "csv_format_refined": "unknown_requires_manual_review",
            "header_line_idx": int(header_line_idx),
            "time_column_name": UNKNOWN,
            "n_rows": len(df),
            "time_parse_method": "unresolved_no_time_column",
            "time_monotonic_status": "unresolved",
            "time_unwrap_count": np.nan,
            "time_reconstructed_from_row_index": False,
            "dt_median_s": np.nan,
            "dt_min_s": np.nan,
            "dt_max_s": np.nan,
            "t_start_s": np.nan,
            "t_end_s": np.nan,
            "duration_s": np.nan,
            "notes": f"columns={df.columns.tolist()}",
        }

    time_col = time_cols[0]
    raw_ts = df[time_col]

    parsed = np.array([parse_day22_timestamp_to_seconds(x) for x in raw_ts], dtype=float)
    t_unwrapped, n_unwrap, rollover_period_s = unwrap_time_if_needed(parsed, raw_ts)

    finite = np.isfinite(t_unwrapped)
    t_valid = t_unwrapped[finite]

    if len(t_valid) >= 2:
        dt = np.diff(t_valid)
        dt_median = float(np.nanmedian(dt))
        dt_min = float(np.nanmin(dt))
        dt_max = float(np.nanmax(dt))
        monotonic = bool(np.all(dt >= -1e-9))
        if monotonic:
            monotonic_status = "monotonic_after_parse_or_unwrap"
        else:
            monotonic_status = "non_monotonic_after_parse"
    else:
        dt = np.array([], dtype=float)
        dt_median = np.nan
        dt_min = np.nan
        dt_max = np.nan
        monotonic_status = "unresolved_too_few_finite_time_values"

    # Refined CSV format classification.
    # If header starts at line 0 and file has no NGU201 metadata rows, treat as processed.
    first_lines = path.read_text(errors="ignore").splitlines()[:5]
    has_device_metadata = any(line.startswith("#Device") for line in first_lines)

    if header_line_idx == 0 and not has_device_metadata:
        csv_format_refined = "processed_1Hz_aligned_no_metadata"
    elif header_info["csv_format"] == "NGU201_LOG_raw":
        csv_format_refined = "NGU201_LOG_raw"
    else:
        csv_format_refined = header_info["csv_format"]

    # Decide if reconstruction is needed.
    # If parsed/unwrap gives monotonic time with median dt around 1s, no reconstruction.
    reconstructed = False
    time_parse_method = "parsed_timestamp_with_unwrap"

    if len(t_valid) < 2 or monotonic_status != "monotonic_after_parse_or_unwrap":
        reconstructed = True
        time_parse_method = "reconstructed_from_row_index_1Hz_due_to_unresolved_time"
        t_valid = np.arange(len(df), dtype=float)
        dt = np.diff(t_valid)
        dt_median = float(np.nanmedian(dt)) if len(dt) else np.nan
        dt_min = float(np.nanmin(dt)) if len(dt) else np.nan
        dt_max = float(np.nanmax(dt)) if len(dt) else np.nan
        monotonic_status = "monotonic_reconstructed_from_row_index"

    return {
        "file_name": path.name,
        "csv_format_refined": csv_format_refined,
        "header_line_idx": int(header_line_idx),
        "time_column_name": str(time_col),
        "n_rows": len(df),
        "time_parse_method": time_parse_method,
        "time_monotonic_status": monotonic_status,
        "time_unwrap_count": int(n_unwrap),
        "time_rollover_period_s": float(rollover_period_s),
        "time_reconstructed_from_row_index": bool(reconstructed),
        "dt_median_s": dt_median,
        "dt_min_s": dt_min,
        "dt_max_s": dt_max,
        "t_start_s": float(t_valid[0]) if len(t_valid) else np.nan,
        "t_end_s": float(t_valid[-1]) if len(t_valid) else np.nan,
        "duration_s": float(t_valid[-1] - t_valid[0]) if len(t_valid) else np.nan,
        "notes": (
            f"original_csv_format={header_info['csv_format']}; "
            f"source_date={header_info['source_date']}; "
            f"start_time={header_info['start_time']}"
        ),
    }


timebase_rows = []
for path in sorted(RAW_DIR_DAY22A.glob("*.csv")):
    timebase_rows.append(inspect_day22_file_timebase(path))

df_day22A_timebase_audit = pd.DataFrame(timebase_rows)
df_day22A_timebase_audit.to_csv(OUT_DAY22A_TIMEBASE_AUDIT, index=False)

print(f"[OK] Wrote Day22A timebase audit: {OUT_DAY22A_TIMEBASE_AUDIT}")
print(df_day22A_timebase_audit.to_string(index=False))

# Hard guard: all files must have usable monotonic time after parse/unwrap or explicit reconstruction.
bad_time = df_day22A_timebase_audit[
    ~df_day22A_timebase_audit["time_monotonic_status"].isin([
        "monotonic_after_parse_or_unwrap",
        "monotonic_reconstructed_from_row_index",
    ])
]

if len(bad_time) > 0:
    raise ValueError(
        "At least one Day22A file has unresolved timebase:\n"
        + bad_time.to_string(index=False)
    )

print("[OK] Cell 0B timebase audit passed.")
print("[OK] No trajectory processing, Q integration, event detection, or verdict performed.")

[OK] Wrote Day22A timebase audit: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step0B_timebase_audit.csv
                          file_name                csv_format_refined  header_line_idx time_column_name  n_rows            time_parse_method           time_monotonic_status  time_unwrap_count  time_rollover_period_s  time_reconstructed_from_row_index  dt_median_s  dt_min_s  dt_max_s  t_start_s  t_end_s  duration_s                                                                                                 notes
MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv processed_1Hz_aligned_no_metadata                0        Timestamp   13257 parsed_timestamp_with_unwrap monotonic_after_parse_or_unwrap                  0                 86400.0                              False          1.0       1.0       1.0        1.0  13257.0     13256.0 original_csv_format=NGU201_LOG_raw; source_date=unknown_not_recorded; start_time=unknown_not_recorded
 MJ1_0p3C_0p4C_10tau_NGU201_raw.csv processed_1Hz_al

In [5]:
# Day22A Cell 1 — audit constants, output paths, schemas, and guards
#
# Purpose:
# - Freeze Day22A paths, schema, and helper functions
# - Extend Day21A-style inventory with CSV-format and timebase provenance
# - Preserve Day21A audit discipline
#
# Explicitly NOT done here:
# - No trajectory loading
# - No Q integration
# - No Vmax / AC-off detection
# - No Δt calculation
# - No verdict

# =============================================================================
# 1. Day22A paths
# =============================================================================

RAW_DIR_DAY22A = DATA_DIR / "raw_mj1_ngu201_day22A_0p3C_0p4C"
METADATA_DIR = DATA_DIR / "metadata"

DAY22A_GROUP_ID = "MJ1_0p3C_0p4C"
DAY22A_NOTEBOOK_NAME = "26_day22A_MJ1_low_amplitude_0p3C_0p4C_audit.ipynb"

OUT_DAY22A_FORMAT_INVENTORY = DATA_DIR / "day22A_step0A_raw_csv_format_inventory.csv"
OUT_DAY22A_TIMEBASE_AUDIT = DATA_DIR / "day22A_step0B_timebase_audit.csv"
OUT_DAY22A_FILE_INVENTORY = DATA_DIR / "day22A_step0_MJ1_0p3C_0p4C_file_inventory.csv"
OUT_DAY22A_AUDIT_CONTRACT_JSON = DATA_DIR / "day22A_audit_contract_schema_thresholds.json"

DAY22A_MANUAL_METADATA_CSV = METADATA_DIR / "day22A_MJ1_0p3C_0p4C_manual_metadata.csv"

EXPECTED_DAY22A_RAW_FILES = [
    "MJ1_0p3C_DC_NGU201_raw.csv",
    "MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv",
    "MJ1_0p3C_0p4C_1tau_NGU201_raw.csv",
    "MJ1_0p3C_0p4C_10tau_NGU201_raw.csv",
]

# =============================================================================
# 2. Day22A protocol constants
# =============================================================================

DAY22A_DC_C = 0.3
DAY22A_AC_C = 0.4

DAY22A_PROTOCOL_ROLE_DC = "DC_reference"
DAY22A_PROTOCOL_ROLE_DCAC = "DCAC"

DAY22A_PROTOCOL_LABELS = {
    "MJ1_0p3C_DC_NGU201_raw.csv": {
        "protocol_label": "0.3C DC",
        "protocol_role": DAY22A_PROTOCOL_ROLE_DC,
        "DC_C": 0.3,
        "AC_C": 0.0,
        "m_tau": np.nan,
        "frequency_Hz": 0.0,
        "tau_eff_s": np.nan,
        "candidate_for_DC_reference": True,
        "candidate_for_DCAC": False,
    },
    "MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv": {
        "protocol_label": "0.3C+0.4C 0.1tau",
        "protocol_role": DAY22A_PROTOCOL_ROLE_DCAC,
        "DC_C": 0.3,
        "AC_C": 0.4,
        "m_tau": 0.1,
        "candidate_for_DC_reference": False,
        "candidate_for_DCAC": True,
    },
    "MJ1_0p3C_0p4C_1tau_NGU201_raw.csv": {
        "protocol_label": "0.3C+0.4C 1tau",
        "protocol_role": DAY22A_PROTOCOL_ROLE_DCAC,
        "DC_C": 0.3,
        "AC_C": 0.4,
        "m_tau": 1.0,
        "candidate_for_DC_reference": False,
        "candidate_for_DCAC": True,
    },
    "MJ1_0p3C_0p4C_10tau_NGU201_raw.csv": {
        "protocol_label": "0.3C+0.4C 10tau",
        "protocol_role": DAY22A_PROTOCOL_ROLE_DCAC,
        "DC_C": 0.3,
        "AC_C": 0.4,
        "m_tau": 10.0,
        "candidate_for_DC_reference": False,
        "candidate_for_DCAC": True,
    },
}

for file_name, meta in DAY22A_PROTOCOL_LABELS.items():
    if np.isfinite(meta.get("m_tau", np.nan)):
        tau_eff = compute_tau_eff_s(TAU_LABEL_S, meta["m_tau"])
        meta["tau_eff_s"] = tau_eff
        meta["frequency_Hz"] = compute_frequency_hz_from_tau_eff(tau_eff)

# =============================================================================
# 3. Day22A inherited audit thresholds
# =============================================================================

# Keep Day21A constants unless explicitly justified.
Q80_NOMINAL_FRACTION_OF_Q_NOM = 0.80
Q90_NOMINAL_FRACTION_OF_Q_NOM = 0.90

Q80_NOMINAL_AH = Q80_NOMINAL_FRACTION_OF_Q_NOM * Q_NOM_AH
Q90_NOMINAL_AH = Q90_NOMINAL_FRACTION_OF_Q_NOM * Q_NOM_AH

FINAL_Q_DIFF_THRESHOLD_AH = 0.010

SEGMENT_A_Q_LO_AH = 0.050
Q_GRID_STEP_AH = 0.010
Q_GRID_MIN_COUNT_SEGMENT_A = 30

Q_SEGMENTB_DEGENERATE_TOLERANCE_AH = 0.001

# Same as Day21A. This is an experimental residual-floor estimate from Day21A,
# not a repeat-based Day22A floor.
MJ1_FLOOR_MAX_ABS_S = 1.35
MJ1_FLOOR_TYPE = "single_condition_experimental_residual_floor_estimate_from_Day21A"
MJ1_FLOOR_N = 1

SEG_A_FLOOR_COMPATIBLE_THRESHOLD_S = 2.0 * MJ1_FLOOR_MAX_ABS_S
SEG_A_REOPEN_THRESHOLD_S = 5.0 * MJ1_FLOOR_MAX_ABS_S

LATE_CV_PRESERVATION_THRESHOLD_S = SEG_A_FLOOR_COMPATIBLE_THRESHOLD_S

# Day22A-specific resolution language
DAY22A_RESOLUTION_STATUS_NO_REPEAT = "no_independent_repeat_based_experimental_noise_floor"
DAY22A_LOW_AMPLITUDE_VERDICT_LIMITATION = (
    "effect_size_interpreted_relative_to_audit_resolution_not_strict_disappearance"
)

# =============================================================================
# 4. Segment and verdict labels
# =============================================================================

SEGMENT_A = "A_shared_prescribed_current"
SEGMENT_B = "B_voltage_boundary_control_state_split"
SEGMENT_D = "D_late_CV_feedback"
SEGMENT_OUTSIDE = "outside_common_Q_window"
SEGMENT_UNRESOLVED = "segment_unresolved"

SEGMENT_FRAMEWORK_OK = "segment_framework_ok"
SEGMENT_FRAMEWORK_ORDERING_VIOLATED = "ordering_violated"
SEGMENT_FRAMEWORK_AC_OFF_PRECEDES_VMAX = "AC_off_precedes_Vmax"
SEGMENT_FRAMEWORK_SEGMENT_B_DEGENERATE = "segment_B_degenerate"
SEGMENT_FRAMEWORK_UNRESOLVED = "segment_framework_unresolved"

ORDERING_EXPECTED = "expected_Q_segmentB_start_lt_Q_DC_Vmax"
ORDERING_DEGENERATE = "segment_B_degenerate_Q_segmentB_start_eq_Q_DC_Vmax"
ORDERING_VIOLATED = "ordering_violated"
ORDERING_UNRESOLVED = "ordering_unresolved"

FINAL_Q_CONSISTENT = "final_Q_consistent"
FINAL_Q_MISMATCH_WARNING = "final_Q_mismatch_warning"
FINAL_Q_UNRESOLVED = "final_Q_unresolved"

GEOM_PHASE_VERIFIED = "verified"
GEOM_PHASE_ESTIMATED = "estimated_from_current_waveform"
GEOM_PHASE_UNRESOLVED = "unresolved"

ABOVE_FLOOR_NO = "floor_compatible"
ABOVE_FLOOR_INTERMEDIATE = "intermediate_between_floor_and_reopen_threshold"
ABOVE_FLOOR_YES = "above_floor"
ABOVE_FLOOR_SPIKE = "spike_or_transition_artifact"
ABOVE_FLOOR_UNRESOLVED = "above_floor_unresolved"

LATE_CV_SATISFIED = "satisfied"
LATE_CV_NOT_SATISFIED = "not_satisfied"
LATE_CV_NOT_REQUIRED = "not_required_no_segmentD_anchor"
LATE_CV_UNRESOLVED = "unresolved"

# =============================================================================
# 5. Day22A file inventory schema
# =============================================================================

DAY22A_INVENTORY_SCHEMA = [
    "file_path",
    "file_name",
    "file_mtime",
    "file_size_kb",
    "read_ok",

    "source_type",
    "cell_id",
    "source_session_date",

    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "frequency_Hz",
    "m_tau",
    "tau_label_s",
    "tau_eff_s",
    "phase_convention",

    "Vmax_V",
    "I_cutoff_A",
    "I_charge_onset_threshold_A",
    "Q_nom_Ah",

    "voltage_source",
    "current_source",
    "sampling_rate_Hz",

    "ambient_temperature_C",
    "temperature_control_type",
    "temperature_sensor_type",
    "temperature_sensor_placement",
    "temperature_data_source",
    "temperature_alignment_method",
    "temperature_to_NGU201_alignment_required",
    "T_surface_max_C",
    "T_surface_mean_C",

    "timebase_source",
    "time_alignment_method",
    "voltage_current_alignment_status",

    "csv_format",
    "csv_format_refined",
    "header_line_idx",
    "time_column_name",
    "time_parse_method",
    "time_reconstructed_from_row_index",
    "time_monotonic_status",
    "time_unwrap_count",
    "time_rollover_period_s",
    "dt_median_s",
    "dt_min_s",
    "dt_max_s",

    "has_time",
    "has_voltage",
    "has_current",
    "has_temperature",
    "has_stage_marker",
    "has_AC_off_marker",

    "candidate_for_DC_reference",
    "candidate_for_DCAC",
    "notes",
]

DAY22A_INVENTORY_NUMERIC_COLUMNS = [
    "file_size_kb",
    "DC_C",
    "AC_C",
    "frequency_Hz",
    "m_tau",
    "tau_label_s",
    "tau_eff_s",
    "Vmax_V",
    "I_cutoff_A",
    "I_charge_onset_threshold_A",
    "Q_nom_Ah",
    "sampling_rate_Hz",
    "ambient_temperature_C",
    "T_surface_max_C",
    "T_surface_mean_C",
    "header_line_idx",
    "time_unwrap_count",
    "time_rollover_period_s",
    "dt_median_s",
    "dt_min_s",
    "dt_max_s",
]

# =============================================================================
# 6. Generic guards and helpers
# =============================================================================

def assert_unique_columns_day22(schema: list[str], schema_name: str) -> None:
    duplicates = sorted({c for c in schema if schema.count(c) > 1})
    if duplicates:
        raise ValueError(f"{schema_name} contains duplicate columns: {duplicates}")


def assert_exact_schema_day22(df: pd.DataFrame, schema: list[str], schema_name: str) -> None:
    actual = list(df.columns)
    expected = list(schema)
    if actual != expected:
        missing = [c for c in expected if c not in actual]
        extra = [c for c in actual if c not in expected]
        raise ValueError(
            f"{schema_name} mismatch.\n"
            f"Missing: {missing}\n"
            f"Extra: {extra}"
        )


def assert_numeric_columns_in_schema_day22(
    numeric_columns: list[str],
    schema: list[str],
    schema_name: str,
) -> None:
    missing = [c for c in numeric_columns if c not in schema]
    if missing:
        raise ValueError(
            f"{schema_name} numeric columns not in schema: {missing}"
        )


def is_finite_number(x: Any) -> bool:
    try:
        return bool(np.isfinite(float(x)))
    except Exception:
        return False


def parse_bool_strict_day22(x: Any) -> bool:
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    if isinstance(x, (int, np.integer)) and x in [0, 1]:
        return bool(x)
    if isinstance(x, str):
        s = x.strip().lower()
        if s == "true":
            return True
        if s == "false":
            return False
    raise ValueError(f"Cannot parse strict bool from {x!r}")


def fill_unknown_strings_and_nan_numeric_day22(
    df: pd.DataFrame,
    numeric_columns: list[str],
    unknown: str = UNKNOWN,
) -> pd.DataFrame:
    out = df.copy()
    numeric_set = set(numeric_columns)

    for col in out.columns:
        if col in numeric_set:
            out[col] = pd.to_numeric(out[col], errors="coerce")
        else:
            out[col] = out[col].where(out[col].notna(), unknown)
            out[col] = out[col].replace("", unknown)

    return out


def assert_candidate_roles_mutually_exclusive_day22(df_inventory: pd.DataFrame) -> None:
    required = ["candidate_for_DC_reference", "candidate_for_DCAC"]
    missing = [c for c in required if c not in df_inventory.columns]
    if missing:
        raise ValueError(f"Inventory missing candidate columns: {missing}")

    dc_ref = df_inventory["candidate_for_DC_reference"].map(parse_bool_strict_day22)
    dcac = df_inventory["candidate_for_DCAC"].map(parse_bool_strict_day22)

    both = dc_ref & dcac
    if both.any():
        bad = df_inventory.loc[both, ["file_name", "protocol_label", *required]]
        raise ValueError(
            "candidate_for_DC_reference and candidate_for_DCAC must be mutually exclusive.\n"
            + bad.to_string(index=False)
        )


def infer_day22_protocol_from_filename(file_name: str) -> dict:
    if file_name not in DAY22A_PROTOCOL_LABELS:
        return {
            "protocol_label": UNKNOWN,
            "protocol_role": UNKNOWN,
            "DC_C": np.nan,
            "AC_C": np.nan,
            "m_tau": np.nan,
            "frequency_Hz": np.nan,
            "tau_eff_s": np.nan,
            "candidate_for_DC_reference": False,
            "candidate_for_DCAC": False,
        }

    return DAY22A_PROTOCOL_LABELS[file_name].copy()


# =============================================================================
# 7. Contract JSON
# =============================================================================

def get_git_head_day22(repo: Path) -> str:
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=repo,
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception as exc:
        return f"unknown_not_recorded:{type(exc).__name__}"


GIT_HEAD_DAY22A = get_git_head_day22(REPO)
RUN_TIMESTAMP_UTC_DAY22A = datetime.now(timezone.utc).isoformat()

DAY22A_AUDIT_CONTRACT = {
    "notebook": DAY22A_NOTEBOOK_NAME,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC_DAY22A,
    "git_head": GIT_HEAD_DAY22A,
    "group_id": DAY22A_GROUP_ID,
    "inherited_from": "Day21A MJ1 full-protocol segmentation audit",
    "day21A_closure_reference": "docs/day21A_close.md",
    "day22A_plan_reference": "docs/day22A_plan_low_amplitude_mj1_audit.md",
    "raw_dir": str(RAW_DIR_DAY22A),
    "expected_files": EXPECTED_DAY22A_RAW_FILES,
    "strict_net_convention": {
        "signed_current": True,
        "rectification": False,
        "cummax": False,
        "first_passage": True,
    },
    "low_amplitude_resolution_limitation": {
        "repeat_based_noise_floor_available": False,
        "formal_disappearance_claim_allowed": False,
        "resolution_status": DAY22A_RESOLUTION_STATUS_NO_REPEAT,
        "verdict_limitation": DAY22A_LOW_AMPLITUDE_VERDICT_LIMITATION,
    },
    "csv_format_requirements": {
        "supported_formats": [
            "NGU201_LOG_raw",
            "processed_1Hz_aligned_no_metadata",
        ],
        "no_silent_time_axis_repair": True,
        "required_timebase_fields": [
            "csv_format_refined",
            "time_parse_method",
            "time_reconstructed_from_row_index",
            "time_monotonic_status",
            "time_unwrap_count",
        ],
    },
    "schemas": {
        "inventory_schema": DAY22A_INVENTORY_SCHEMA,
        "inventory_numeric_columns": DAY22A_INVENTORY_NUMERIC_COLUMNS,
    },
}

assert_unique_columns_day22(DAY22A_INVENTORY_SCHEMA, "DAY22A_INVENTORY_SCHEMA")
assert_numeric_columns_in_schema_day22(
    DAY22A_INVENTORY_NUMERIC_COLUMNS,
    DAY22A_INVENTORY_SCHEMA,
    "DAY22A_INVENTORY_SCHEMA",
)

with open(OUT_DAY22A_AUDIT_CONTRACT_JSON, "w", encoding="utf-8") as f:
    json.dump(DAY22A_AUDIT_CONTRACT, f, indent=2, ensure_ascii=False)

print("[OK] Day22A Cell 1 constants/schema frozen.")
print(f"[OK] Git HEAD: {GIT_HEAD_DAY22A}")
print(f"[OK] Audit contract JSON: {OUT_DAY22A_AUDIT_CONTRACT_JSON}")
print(f"[OK] Inventory schema columns: {len(DAY22A_INVENTORY_SCHEMA)}")
print(f"[OK] Expected files: {len(EXPECTED_DAY22A_RAW_FILES)}")
print("[OK] No trajectory processing performed in Cell 1.")

[OK] Day22A Cell 1 constants/schema frozen.
[OK] Git HEAD: c40e2dc2748b55e3f63f25101723ae13eaf65211
[OK] Audit contract JSON: /Users/louislu/pybamm-dcac-superimposed/data/day22A_audit_contract_schema_thresholds.json
[OK] Inventory schema columns: 57
[OK] Expected files: 4
[OK] No trajectory processing performed in Cell 1.


In [8]:
# Day22A Cell 2 — file inventory with CSV-format and timebase provenance
#
# Purpose:
# - Build Day22A file-level inventory
# - Merge Cell 0B timebase audit
# - Infer protocol metadata from locked filenames
# - Create / merge manual metadata template for temperature summaries
# - Write day22A_step0_MJ1_0p3C_0p4C_file_inventory.csv
#
# Explicitly NOT done here:
# - No trajectory loading
# - No charge-onset trimming
# - No Q integration
# - No Vmax / AC-off detection
# - No Δt calculation
# - No verdict

# =============================================================================
# 2.1 Output paths and required inputs
# =============================================================================

if not RAW_DIR_DAY22A.exists():
    raise FileNotFoundError(f"Day22A raw directory not found: {RAW_DIR_DAY22A}")

if not OUT_DAY22A_TIMEBASE_AUDIT.exists():
    raise FileNotFoundError(
        f"Day22A timebase audit not found: {OUT_DAY22A_TIMEBASE_AUDIT}\n"
        "Run Cell 0B first."
    )

df_timebase = pd.read_csv(OUT_DAY22A_TIMEBASE_AUDIT)

required_timebase_cols = [
    "file_name",
    "csv_format_refined",
    "header_line_idx",
    "time_column_name",
    "time_parse_method",
    "time_monotonic_status",
    "time_unwrap_count",
    "time_rollover_period_s",
    "time_reconstructed_from_row_index",
    "dt_median_s",
    "dt_min_s",
    "dt_max_s",
]

missing_cols = [c for c in required_timebase_cols if c not in df_timebase.columns]
if missing_cols:
    raise ValueError(f"Timebase audit missing required columns: {missing_cols}")

csv_paths = sorted(RAW_DIR_DAY22A.glob("*.csv"))
found_names = [p.name for p in csv_paths]

missing_expected = [name for name in EXPECTED_DAY22A_RAW_FILES if name not in found_names]
unexpected_files = [name for name in found_names if name not in EXPECTED_DAY22A_RAW_FILES]

print(f"[scan] RAW_DIR_DAY22A = {RAW_DIR_DAY22A}")
print(f"[scan] found CSV files = {len(csv_paths)}")
print(f"[scan] expected files missing = {missing_expected}")
print(f"[scan] unexpected CSV files = {unexpected_files}")

if missing_expected:
    raise FileNotFoundError(
        "Missing expected Day22A raw files:\n" + "\n".join(missing_expected)
    )


# =============================================================================
# 2.2 Helper functions
# =============================================================================

def get_timebase_row_day22(file_name: str) -> pd.Series:
    rows = df_timebase[df_timebase["file_name"] == file_name]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one timebase row for {file_name}, found {len(rows)}")
    return rows.iloc[0]


def detect_header_columns_day22(path: Path, header_line_idx: int, delimiter: str = ",") -> list[str]:
    """
    Read only the detected header row. This is not trajectory loading.
    """
    try:
        lines = path.read_text(errors="ignore").splitlines()
        if header_line_idx < 0 or header_line_idx >= len(lines):
            return []
        return [p.strip() for p in lines[header_line_idx].split(delimiter)]
    except Exception:
        return []


def column_flags_day22(columns: list[str]) -> dict[str, bool]:
    lowered = [str(c).lower().strip() for c in columns]

    has_time = any(
        c in {"timestamp", "time", "t", "t_s", "time_s", "time[s]", "t[s]"}
        or "time" in c
        for c in lowered
    )

    has_voltage = any(
        c in {"u", "u_v", "u1[v]", "voltage", "voltage_v", "u[v]", "v"}
        or "volt" in c
        or "[v]" in c
        for c in lowered
    )

    has_current = any(
        c in {"i", "i_a", "i1[a]", "current", "current_a", "i[a]"}
        or "curr" in c
        or "[a]" in c
        for c in lowered
    )

    has_temperature = any(
        ("temp" in c)
        or ("temperature" in c)
        or ("pt100" in c)
        or ("°c" in c)
        or ("[c]" in c)
        for c in lowered
    )

    has_stage_marker = any(
        ("stage" in c)
        or ("mode" in c)
        or ("state" in c)
        or ("step" in c)
        or (c == "cv")
        or (c == "cc")
        for c in lowered
    )

    has_acoff_marker = any(
        ("ac_off" in c)
        or ("acoff" in c)
        or ("arb" in c)
        or ("arbitrary" in c)
        for c in lowered
    )

    return {
        "has_time": bool(has_time),
        "has_voltage": bool(has_voltage),
        "has_current": bool(has_current),
        "has_temperature": bool(has_temperature),
        "has_stage_marker": bool(has_stage_marker),
        "has_AC_off_marker": bool(has_acoff_marker),
    }


def make_day22_inventory_row(path: Path) -> dict:
    stat = path.stat()
    file_name = path.name

    proto = infer_day22_protocol_from_filename(file_name)
    tb = get_timebase_row_day22(file_name)

    header_line_idx = int(tb["header_line_idx"])
    columns = detect_header_columns_day22(path, header_line_idx=header_line_idx, delimiter=",")
    flags = column_flags_day22(columns)

    csv_format_refined = str(tb["csv_format_refined"])

    if csv_format_refined == "NGU201_LOG_raw":
        csv_format = "NGU201_LOG_raw"
    elif csv_format_refined == "processed_1Hz_aligned_no_metadata":
        csv_format = "processed_1Hz_aligned_no_metadata"
    else:
        csv_format = str(csv_format_refined)

    # Sampling rate from timebase audit median dt
    dt_median_s = tb["dt_median_s"]
    if is_finite_number(dt_median_s) and float(dt_median_s) > 0:
        sampling_rate_Hz = 1.0 / float(dt_median_s)
    else:
        sampling_rate_Hz = np.nan

    # Source date is not available for processed no-metadata files.
    source_session_date = UNKNOWN

    # Preserve known dates from timebase notes if available is intentionally not parsed here.
    # Manual metadata can override source_session_date later.
    if csv_format_refined == "NGU201_LOG_raw":
        # For NGU201 raw, source date is embedded in notes but not kept as a separate column by Cell 0B.
        # Keep unknown unless manual metadata supplies it.
        source_session_date = UNKNOWN

    row = {
        "file_path": str(path),
        "file_name": file_name,
        "file_mtime": datetime.fromtimestamp(stat.st_mtime).isoformat(),
        "file_size_kb": stat.st_size / 1024.0,
        "read_ok": True,

        "source_type": SOURCE_TYPE_MJ1,
        "cell_id": CELL_ID,
        "source_session_date": source_session_date,

        "protocol_label": proto["protocol_label"],
        "protocol_role": proto["protocol_role"],
        "DC_C": proto["DC_C"],
        "AC_C": proto["AC_C"],
        "frequency_Hz": proto["frequency_Hz"],
        "m_tau": proto["m_tau"],
        "tau_label_s": TAU_LABEL_S,
        "tau_eff_s": proto["tau_eff_s"],
        "phase_convention": PHASE_CONVENTION,

        "Vmax_V": VMAX_V,
        "I_cutoff_A": I_CUTOFF_A,
        "I_charge_onset_threshold_A": I_CHARGE_ONSET_THRESHOLD_A,
        "Q_nom_Ah": Q_NOM_AH,

        "voltage_source": DEFAULT_VOLTAGE_SOURCE,
        "current_source": DEFAULT_CURRENT_SOURCE,
        "sampling_rate_Hz": sampling_rate_Hz,

        "ambient_temperature_C": AMBIENT_TEMPERATURE_C,
        "temperature_control_type": TEMPERATURE_CONTROL_TYPE,
        "temperature_sensor_type": TEMPERATURE_SENSOR_TYPE,
        "temperature_sensor_placement": TEMPERATURE_SENSOR_PLACEMENT,
        "temperature_data_source": TEMPERATURE_DATA_SOURCE,
        "temperature_alignment_method": TEMPERATURE_ALIGNMENT_METHOD,
        "temperature_to_NGU201_alignment_required": TEMPERATURE_TO_NGU201_ALIGNMENT_REQUIRED,
        "T_surface_max_C": np.nan,
        "T_surface_mean_C": np.nan,

        "timebase_source": (
            "processed_monotonic_1Hz_timebase"
            if csv_format_refined == "processed_1Hz_aligned_no_metadata"
            else DEFAULT_TIMEBASE_SOURCE
        ),
        "time_alignment_method": (
            "processed_same_record_monotonic_timebase"
            if csv_format_refined == "processed_1Hz_aligned_no_metadata"
            else DEFAULT_TIME_ALIGNMENT_METHOD
        ),
        "voltage_current_alignment_status": DEFAULT_VOLTAGE_CURRENT_ALIGNMENT_STATUS,

        "csv_format": csv_format,
        "csv_format_refined": csv_format_refined,
        "header_line_idx": header_line_idx,
        "time_column_name": tb["time_column_name"],
        "time_parse_method": tb["time_parse_method"],
        "time_reconstructed_from_row_index": tb["time_reconstructed_from_row_index"],
        "time_monotonic_status": tb["time_monotonic_status"],
        "time_unwrap_count": tb["time_unwrap_count"],
        "time_rollover_period_s": tb["time_rollover_period_s"],
        "dt_median_s": tb["dt_median_s"],
        "dt_min_s": tb["dt_min_s"],
        "dt_max_s": tb["dt_max_s"],

        **flags,

        "candidate_for_DC_reference": proto["candidate_for_DC_reference"],
        "candidate_for_DCAC": proto["candidate_for_DCAC"],
        "notes": f"columns={columns}; timebase_notes={tb.get('notes', UNKNOWN)}",
    }

    return row


# =============================================================================
# 2.3 Build inventory rows
# =============================================================================

rows = [make_day22_inventory_row(path) for path in csv_paths]
df_day22_inventory = pd.DataFrame(rows)


# =============================================================================
# 2.4 Manual metadata template / merge
# =============================================================================

DAY22A_MANUAL_METADATA_COLUMNS = [
    "file_name",
    "source_session_date",
    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "frequency_Hz",
    "m_tau",
    "phase_convention",
    "sampling_rate_Hz",
    "ambient_temperature_C",
    "temperature_control_type",
    "temperature_sensor_type",
    "temperature_sensor_placement",
    "temperature_data_source",
    "temperature_alignment_method",
    "temperature_to_NGU201_alignment_required",
    "T_surface_max_C",
    "T_surface_mean_C",
    "notes",
]

if not DAY22A_MANUAL_METADATA_CSV.exists():
    df_template = df_day22_inventory[DAY22A_MANUAL_METADATA_COLUMNS].copy()
    df_template.to_csv(DAY22A_MANUAL_METADATA_CSV, index=False)
    print(f"[metadata] Created Day22A manual metadata template: {DAY22A_MANUAL_METADATA_CSV}")
else:
    df_manual = pd.read_csv(DAY22A_MANUAL_METADATA_CSV)

    missing_manual_cols = [
        c for c in DAY22A_MANUAL_METADATA_COLUMNS if c not in df_manual.columns
    ]
    if missing_manual_cols:
        raise ValueError(
            f"Manual metadata CSV missing columns: {missing_manual_cols}\n"
            f"File: {DAY22A_MANUAL_METADATA_CSV}"
        )

    editable_cols = [c for c in DAY22A_MANUAL_METADATA_COLUMNS if c != "file_name"]

    df_day22_inventory = df_day22_inventory.merge(
        df_manual[["file_name", *editable_cols]],
        on="file_name",
        how="left",
        suffixes=("", "_manual"),
    )

    for col in editable_cols:
        manual_col = f"{col}_manual"
        if manual_col in df_day22_inventory.columns:
            manual_values = df_day22_inventory[manual_col]
            if manual_values.dtype == object:
                manual_values = manual_values.replace("", np.nan)
            df_day22_inventory[col] = manual_values.where(
                manual_values.notna(),
                df_day22_inventory[col],
            )
            df_day22_inventory = df_day22_inventory.drop(columns=[manual_col])

    print(f"[metadata] Merged Day22A manual metadata: {DAY22A_MANUAL_METADATA_CSV}")


# =============================================================================
# 2.5 Enforce schema and guards
# =============================================================================

for col in DAY22A_INVENTORY_SCHEMA:
    if col not in df_day22_inventory.columns:
        df_day22_inventory[col] = (
            np.nan if col in DAY22A_INVENTORY_NUMERIC_COLUMNS else UNKNOWN
        )

df_day22_inventory = df_day22_inventory[DAY22A_INVENTORY_SCHEMA].copy()

df_day22_inventory = fill_unknown_strings_and_nan_numeric_day22(
    df_day22_inventory,
    numeric_columns=DAY22A_INVENTORY_NUMERIC_COLUMNS,
    unknown=UNKNOWN,
)

df_day22_inventory["candidate_for_DC_reference"] = (
    df_day22_inventory["candidate_for_DC_reference"].map(parse_bool_strict_day22)
)
df_day22_inventory["candidate_for_DCAC"] = (
    df_day22_inventory["candidate_for_DCAC"].map(parse_bool_strict_day22)
)

assert_exact_schema_day22(
    df_day22_inventory,
    DAY22A_INVENTORY_SCHEMA,
    "DAY22A_INVENTORY_SCHEMA",
)
assert_candidate_roles_mutually_exclusive_day22(df_day22_inventory)

n_dc_ref = int(df_day22_inventory["candidate_for_DC_reference"].sum())
n_dcac = int(df_day22_inventory["candidate_for_DCAC"].sum())

if n_dc_ref != 1:
    raise ValueError(f"Expected exactly 1 DC reference candidate, found {n_dc_ref}")

if n_dcac != 3:
    raise ValueError(f"Expected exactly 3 DCAC candidates, found {n_dcac}")

core_signal_ok = (
    df_day22_inventory["has_time"].map(parse_bool_strict_day22)
    & df_day22_inventory["has_voltage"].map(parse_bool_strict_day22)
    & df_day22_inventory["has_current"].map(parse_bool_strict_day22)
)

if not core_signal_ok.all():
    bad = df_day22_inventory.loc[
        ~core_signal_ok,
        ["file_name", "has_time", "has_voltage", "has_current", "notes"],
    ]
    raise ValueError(
        "At least one Day22A file lacks required time/voltage/current columns:\n"
        + bad.to_string(index=False)
    )

# Hard guard: timebase must have passed Cell 0B.
valid_timebase_status = {
    "monotonic_after_parse_or_unwrap",
    "monotonic_reconstructed_from_row_index",
}

bad_time = df_day22_inventory[
    ~df_day22_inventory["time_monotonic_status"].isin(valid_timebase_status)
]

if len(bad_time) > 0:
    raise ValueError(
        "At least one Day22A file has unresolved timebase:\n"
        + bad_time[[
            "file_name",
            "csv_format_refined",
            "time_parse_method",
            "time_monotonic_status",
            "notes",
        ]].to_string(index=False)
    )

df_day22_inventory.to_csv(OUT_DAY22A_FILE_INVENTORY, index=False)

print(f"[OK] Wrote Day22A file inventory: {OUT_DAY22A_FILE_INVENTORY}")
print(f"[OK] inventory shape = {df_day22_inventory.shape}")
print(f"[OK] DC reference candidates = {n_dc_ref}")
print(f"[OK] DCAC candidates = {n_dcac}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "m_tau",
    "frequency_Hz",
    "csv_format_refined",
    "time_parse_method",
    "time_monotonic_status",
    "time_reconstructed_from_row_index",
    "dt_median_s",
    "dt_min_s",
    "dt_max_s",
    "sampling_rate_Hz",
    "T_surface_max_C",
    "T_surface_mean_C",
    "candidate_for_DC_reference",
    "candidate_for_DCAC",
]
print(df_day22_inventory[display_cols].to_string(index=False))

print("[OK] Cell 2 Day22A file inventory completed.")
print("[OK] No trajectory loading, Q integration, event detection, or verdict performed.")

[scan] RAW_DIR_DAY22A = /Users/louislu/pybamm-dcac-superimposed/data/raw_mj1_ngu201_day22A_0p3C_0p4C
[scan] found CSV files = 4
[scan] expected files missing = []
[scan] unexpected CSV files = []
[metadata] Merged Day22A manual metadata: /Users/louislu/pybamm-dcac-superimposed/data/metadata/day22A_MJ1_0p3C_0p4C_manual_metadata.csv
[OK] Wrote Day22A file inventory: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step0_MJ1_0p3C_0p4C_file_inventory.csv
[OK] inventory shape = (4, 57)
[OK] DC reference candidates = 1
[OK] DCAC candidates = 3
                          file_name   protocol_label protocol_role  DC_C  AC_C  m_tau  frequency_Hz                csv_format_refined            time_parse_method           time_monotonic_status  time_reconstructed_from_row_index  dt_median_s  dt_min_s  dt_max_s  sampling_rate_Hz  T_surface_max_C  T_surface_mean_C  candidate_for_DC_reference  candidate_for_DCAC
MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv 0.3C+0.4C 0.1tau          DCAC   0.3   0.4    0.1    

In [7]:
# Day22A Cell 2A — patch manual metadata with protocol-level temperature summaries
# Do NOT delete the metadata template after this.

if not DAY22A_MANUAL_METADATA_CSV.exists():
    raise FileNotFoundError(
        f"Manual metadata template not found: {DAY22A_MANUAL_METADATA_CSV}\n"
        "Run Day22A Cell 2 once to create the template, then run this cell."
    )

df_meta22 = pd.read_csv(DAY22A_MANUAL_METADATA_CSV)

temperature_map_day22A = {
    "MJ1_0p3C_DC_NGU201_raw.csv": {
        "T_surface_max_C": 23.553,
        "T_surface_mean_C": 22.610,
    },
    "MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv": {
        "T_surface_max_C": 25.458,
        "T_surface_mean_C": 24.479,
    },
    "MJ1_0p3C_0p4C_1tau_NGU201_raw.csv": {
        "T_surface_max_C": 25.555,
        "T_surface_mean_C": 24.159,
    },
    "MJ1_0p3C_0p4C_10tau_NGU201_raw.csv": {
        "T_surface_max_C": 27.350,
        "T_surface_mean_C": 26.038,
    },
}

for file_name, vals in temperature_map_day22A.items():
    mask = df_meta22["file_name"] == file_name
    if not mask.any():
        raise ValueError(f"File not found in Day22A manual metadata: {file_name}")

    df_meta22.loc[mask, "T_surface_max_C"] = vals["T_surface_max_C"]
    df_meta22.loc[mask, "T_surface_mean_C"] = vals["T_surface_mean_C"]

    df_meta22.loc[mask, "temperature_sensor_type"] = "Pt100"
    df_meta22.loc[mask, "temperature_sensor_placement"] = "axial_cell_surface"
    df_meta22.loc[mask, "temperature_data_source"] = "measured_surface_temperature"
    df_meta22.loc[mask, "temperature_alignment_method"] = "segment_level_summary_only"
    df_meta22.loc[mask, "temperature_to_NGU201_alignment_required"] = False
    df_meta22.loc[mask, "temperature_control_type"] = "ambient_lab_no_chamber"
    df_meta22.loc[mask, "ambient_temperature_C"] = 20.0

df_meta22.to_csv(DAY22A_MANUAL_METADATA_CSV, index=False)

print(f"[OK] Patched Day22A manual metadata: {DAY22A_MANUAL_METADATA_CSV}")
print(df_meta22[[
    "file_name",
    "T_surface_max_C",
    "T_surface_mean_C",
    "temperature_sensor_type",
    "temperature_alignment_method",
]].to_string(index=False))

[OK] Patched Day22A manual metadata: /Users/louislu/pybamm-dcac-superimposed/data/metadata/day22A_MJ1_0p3C_0p4C_manual_metadata.csv
                          file_name  T_surface_max_C  T_surface_mean_C temperature_sensor_type temperature_alignment_method
MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv           25.458            24.479                   Pt100   segment_level_summary_only
 MJ1_0p3C_0p4C_10tau_NGU201_raw.csv           27.350            26.038                   Pt100   segment_level_summary_only
  MJ1_0p3C_0p4C_1tau_NGU201_raw.csv           25.555            24.159                   Pt100   segment_level_summary_only
         MJ1_0p3C_DC_NGU201_raw.csv           23.553            22.610                   Pt100   segment_level_summary_only


In [9]:
# Day22A Cell 3 — trajectory loading and sanity checks
#
# Purpose:
# - Load Day22A trajectories from mixed CSV formats:
#   Type A: NGU201_LOG_raw
#   Type B: processed_1Hz_aligned_no_metadata
# - Standardize to columns:
#   timestamp_raw, t_abs_s, t_s, U_V, I_meas_A, I_Q_A, P_W, DVM_V
# - Trim charge-onset region using I_charge_onset_threshold_A
# - Write load sanity summary
#
# Explicitly NOT done here:
# - No Q integration
# - No Vmax / AC-off detection
# - No Segment A/B/D assignment
# - No Δt computation
# - No verdict

OUT_DAY22A_LOAD_SUMMARY = DATA_DIR / "day22A_step1_MJ1_0p3C_0p4C_loaded_trajectory_sanity.csv"

if not OUT_DAY22A_FILE_INVENTORY.exists():
    raise FileNotFoundError(
        f"Day22A file inventory not found: {OUT_DAY22A_FILE_INVENTORY}\n"
        "Run Day22A Cell 2 first."
    )

df_day22_inventory = pd.read_csv(OUT_DAY22A_FILE_INVENTORY)
assert_exact_schema_day22(
    df_day22_inventory,
    DAY22A_INVENTORY_SCHEMA,
    "DAY22A_INVENTORY_SCHEMA",
)

df_day22_inventory["candidate_for_DC_reference"] = (
    df_day22_inventory["candidate_for_DC_reference"].map(parse_bool_strict_day22)
)
df_day22_inventory["candidate_for_DCAC"] = (
    df_day22_inventory["candidate_for_DCAC"].map(parse_bool_strict_day22)
)

assert_candidate_roles_mutually_exclusive_day22(df_day22_inventory)

required_signal_ok = (
    df_day22_inventory["has_time"].map(parse_bool_strict_day22)
    & df_day22_inventory["has_voltage"].map(parse_bool_strict_day22)
    & df_day22_inventory["has_current"].map(parse_bool_strict_day22)
)

if not required_signal_ok.all():
    bad = df_day22_inventory.loc[
        ~required_signal_ok,
        ["file_name", "has_time", "has_voltage", "has_current", "notes"],
    ]
    raise ValueError(
        "Cannot load trajectories: signal-column inventory failed.\n"
        + bad.to_string(index=False)
    )

print(f"[OK] Loaded Day22A inventory: {OUT_DAY22A_FILE_INVENTORY}")
print(f"[OK] inventory rows = {len(df_day22_inventory)}")


# =============================================================================
# 3.1 Loading helpers
# =============================================================================

def find_charge_onset_index_day22(
    current_q: np.ndarray,
    voltage: np.ndarray,
    threshold_A: float = I_CHARGE_ONSET_THRESHOLD_A,
    min_consecutive: int = I_CHARGE_ONSET_MIN_CONSECUTIVE_SAMPLES,
) -> int:
    """
    Find first sustained charge-positive current onset.

    This trims rest / pre-output / leading NaN rows only.
    It does not establish ARB phase alignment.
    """
    finite = np.isfinite(current_q) & np.isfinite(voltage)
    above = finite & (current_q >= threshold_A)

    n = len(current_q)
    if n < min_consecutive:
        raise ValueError("Trajectory too short for charge-onset detection.")

    for i in range(0, n - min_consecutive + 1):
        if np.all(above[i : i + min_consecutive]):
            return i

    raise ValueError(
        f"No charge onset found with I_Q >= {threshold_A} A "
        f"for {min_consecutive} consecutive samples."
    )


def locate_column_day22(columns: Sequence[str], kind: str) -> str:
    """
    Locate time / voltage / current / power / DVM columns from known variants.
    """
    lowered = {str(c).lower().strip(): c for c in columns}

    if kind == "time":
        candidates = ["timestamp", "time", "t", "t_s", "time_s", "time[s]", "t[s]"]
        for key in candidates:
            if key in lowered:
                return lowered[key]
        for c in columns:
            if "time" in str(c).lower():
                return c

    if kind == "voltage":
        candidates = ["u1[v]", "u[v]", "u_v", "voltage", "voltage_v", "v"]
        for key in candidates:
            if key in lowered:
                return lowered[key]
        for c in columns:
            cl = str(c).lower()
            if "volt" in cl or "[v]" in cl:
                return c

    if kind == "current":
        candidates = ["i1[a]", "i[a]", "i_a", "current", "current_a", "i"]
        for key in candidates:
            if key in lowered:
                return lowered[key]
        for c in columns:
            cl = str(c).lower()
            if "curr" in cl or "[a]" in cl:
                return c

    if kind == "power":
        for key in ["p1[w]", "p[w]", "power", "power_w"]:
            if key in lowered:
                return lowered[key]
        return ""

    if kind == "dvm":
        for key in ["dvm1[v]", "dvm[v]", "dvm"]:
            if key in lowered:
                return lowered[key]
        return ""

    raise ValueError(f"Could not locate {kind} column in columns={list(columns)}")


def build_time_axis_day22(
    timestamp_series: pd.Series,
    time_reconstructed_from_row_index: bool,
) -> tuple[np.ndarray, str, int]:
    """
    Build absolute seconds from timestamp column.

    If inventory says reconstruction is required, use row index at 1 Hz.
    Otherwise parse timestamp and unwrap.
    """
    if bool(time_reconstructed_from_row_index):
        t_abs = np.arange(len(timestamp_series), dtype=float)
        return t_abs, "reconstructed_from_row_index_1Hz", 0

    parsed = np.array(
        [parse_day22_timestamp_to_seconds(x) for x in timestamp_series],
        dtype=float,
    )
    t_unwrapped, n_unwrap, _rollover = unwrap_time_if_needed(parsed, timestamp_series)

    return t_unwrapped, "parsed_timestamp_with_unwrap", int(n_unwrap)


def load_day22_trajectory(inv_row: pd.Series) -> tuple[pd.DataFrame, dict[str, object]]:
    """
    Load one Day22A trajectory according to inventory-provenance fields.

    Returns retained trajectory starting at charge onset.
    No Q integration is performed.
    """
    path = Path(inv_row["file_path"])
    header_line_idx = int(inv_row["header_line_idx"])

    df_raw = pd.read_csv(
        path,
        sep=",",
        skiprows=header_line_idx,
        engine="python",
    )

    # Remove unnamed fully empty trailing columns from processed CSVs.
    drop_cols = []
    for c in df_raw.columns:
        if str(c).lower().startswith("unnamed"):
            if df_raw[c].isna().all():
                drop_cols.append(c)
    if drop_cols:
        df_raw = df_raw.drop(columns=drop_cols)

    time_col = locate_column_day22(df_raw.columns, "time")
    voltage_col = locate_column_day22(df_raw.columns, "voltage")
    current_col = locate_column_day22(df_raw.columns, "current")
    power_col = locate_column_day22(df_raw.columns, "power")
    dvm_col = locate_column_day22(df_raw.columns, "dvm")

    timestamp_raw = df_raw[time_col].astype(str)

    t_abs, time_parse_method_runtime, n_unwrap_runtime = build_time_axis_day22(
        timestamp_series=df_raw[time_col],
        time_reconstructed_from_row_index=parse_bool_strict_day22(
            inv_row["time_reconstructed_from_row_index"]
        ),
    )

    U_V = pd.to_numeric(df_raw[voltage_col], errors="coerce").to_numpy(dtype=float)
    I_meas_A = pd.to_numeric(df_raw[current_col], errors="coerce").to_numpy(dtype=float)

    # NGU201 / processed Day22A convention:
    # charging current is positive.
    I_Q_A = I_meas_A.copy()

    if power_col:
        P_W = pd.to_numeric(df_raw[power_col], errors="coerce").to_numpy(dtype=float)
    else:
        P_W = np.full(len(df_raw), np.nan)

    if dvm_col:
        DVM_V = pd.to_numeric(df_raw[dvm_col], errors="coerce").to_numpy(dtype=float)
    else:
        DVM_V = np.full(len(df_raw), np.nan)

    onset_idx = find_charge_onset_index_day22(
        current_q=I_Q_A,
        voltage=U_V,
        threshold_A=I_CHARGE_ONSET_THRESHOLD_A,
        min_consecutive=I_CHARGE_ONSET_MIN_CONSECUTIVE_SAMPLES,
    )

    t0 = t_abs[onset_idx]
    if not np.isfinite(t0):
        raise ValueError(f"{path.name}: charge onset time is not finite.")

    df = pd.DataFrame(
        {
            "timestamp_raw": timestamp_raw,
            "t_abs_s": t_abs,
            "t_s": t_abs - t0,
            "U_V": U_V,
            "I_meas_A": I_meas_A,
            "I_Q_A": I_Q_A,
            "P_W": P_W,
            "DVM_V": DVM_V,
        }
    )

    df_retained = df.iloc[onset_idx:].reset_index(drop=True)
    df_retained = df_retained[np.isfinite(df_retained["t_s"])].reset_index(drop=True)

    dt = np.diff(df_retained["t_s"].to_numpy(dtype=float))
    dt_finite = dt[np.isfinite(dt)]

    finite_u = np.isfinite(df_retained["U_V"].to_numpy(dtype=float))
    finite_i = np.isfinite(df_retained["I_Q_A"].to_numpy(dtype=float))

    meta = {
        "file_name": path.name,
        "csv_format_refined": inv_row["csv_format_refined"],
        "header_line_idx": header_line_idx,
        "time_column_name": time_col,
        "voltage_column_name": voltage_col,
        "current_column_name": current_col,
        "time_parse_method_runtime": time_parse_method_runtime,
        "time_unwrap_count_runtime": n_unwrap_runtime,
        "n_raw_rows": len(df_raw),
        "n_retained_rows": len(df_retained),
        "charge_onset_idx_raw": int(onset_idx),
        "charge_onset_timestamp_raw": str(timestamp_raw.iloc[onset_idx]),
        "retained_duration_s": (
            float(df_retained["t_s"].iloc[-1])
            if len(df_retained) > 0
            else np.nan
        ),
        "dt_median_s": float(np.nanmedian(dt_finite)) if len(dt_finite) else np.nan,
        "dt_min_s": float(np.nanmin(dt_finite)) if len(dt_finite) else np.nan,
        "dt_max_s": float(np.nanmax(dt_finite)) if len(dt_finite) else np.nan,
        "finite_voltage_fraction": float(np.mean(finite_u)) if len(df_retained) else np.nan,
        "finite_current_fraction": float(np.mean(finite_i)) if len(df_retained) else np.nan,
        "U_min_V": float(np.nanmin(df_retained["U_V"])) if finite_u.any() else np.nan,
        "U_max_V": float(np.nanmax(df_retained["U_V"])) if finite_u.any() else np.nan,
        "I_Q_min_A": float(np.nanmin(df_retained["I_Q_A"])) if finite_i.any() else np.nan,
        "I_Q_max_A": float(np.nanmax(df_retained["I_Q_A"])) if finite_i.any() else np.nan,
        "I_Q_mean_A": float(np.nanmean(df_retained["I_Q_A"])) if finite_i.any() else np.nan,
    }

    return df_retained, meta


# =============================================================================
# 3.2 Load all Day22A trajectories
# =============================================================================

TRAJ22: dict[str, pd.DataFrame] = {}
load_rows = []

for _, inv_row in df_day22_inventory.iterrows():
    file_name = inv_row["file_name"]

    df_traj, meta = load_day22_trajectory(inv_row)

    # Attach protocol metadata
    df_traj["file_name"] = file_name
    df_traj["protocol_label"] = inv_row["protocol_label"]
    df_traj["protocol_role"] = inv_row["protocol_role"]
    df_traj["DC_C"] = inv_row["DC_C"]
    df_traj["AC_C"] = inv_row["AC_C"]
    df_traj["m_tau"] = inv_row["m_tau"]
    df_traj["frequency_Hz"] = inv_row["frequency_Hz"]
    df_traj["phase_convention"] = inv_row["phase_convention"]
    df_traj["csv_format_refined"] = inv_row["csv_format_refined"]

    TRAJ22[file_name] = df_traj

    row = {
        "file_name": file_name,
        "protocol_label": inv_row["protocol_label"],
        "protocol_role": inv_row["protocol_role"],
        "DC_C": inv_row["DC_C"],
        "AC_C": inv_row["AC_C"],
        "m_tau": inv_row["m_tau"],
        "frequency_Hz": inv_row["frequency_Hz"],
        "sampling_rate_Hz_inventory": inv_row["sampling_rate_Hz"],
        **meta,
    }
    load_rows.append(row)

df_day22_load_summary = pd.DataFrame(load_rows)

OUT_DAY22A_LOAD_SUMMARY = DATA_DIR / "day22A_step1_MJ1_0p3C_0p4C_loaded_trajectory_sanity.csv"
df_day22_load_summary.to_csv(OUT_DAY22A_LOAD_SUMMARY, index=False)

print(f"[OK] Loaded Day22A retained trajectories: {len(TRAJ22)}")
print(f"[OK] Wrote Day22A load sanity summary: {OUT_DAY22A_LOAD_SUMMARY}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "csv_format_refined",
    "n_raw_rows",
    "n_retained_rows",
    "charge_onset_idx_raw",
    "charge_onset_timestamp_raw",
    "retained_duration_s",
    "dt_median_s",
    "dt_min_s",
    "dt_max_s",
    "finite_voltage_fraction",
    "finite_current_fraction",
    "U_min_V",
    "U_max_V",
    "I_Q_min_A",
    "I_Q_max_A",
    "I_Q_mean_A",
]
print(df_day22_load_summary[display_cols].to_string(index=False))


# =============================================================================
# 3.3 Hard sanity guards
# =============================================================================

for file_name, df_traj in TRAJ22.items():
    if len(df_traj) < 10:
        raise ValueError(f"{file_name}: retained trajectory has fewer than 10 rows.")

    if not df_traj["t_s"].is_monotonic_increasing:
        raise ValueError(f"{file_name}: retained t_s is not monotonic increasing.")

    if df_traj["U_V"].notna().mean() < 0.95:
        raise ValueError(f"{file_name}: voltage finite fraction below 0.95.")

    if df_traj["I_Q_A"].notna().mean() < 0.95:
        raise ValueError(f"{file_name}: current finite fraction below 0.95.")

print("[OK] Cell 3 Day22A trajectory loading and sanity checks passed.")
print("[OK] No Q integration, event detection, segmentation, Δt computation, or verdict performed.")

[OK] Loaded Day22A inventory: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step0_MJ1_0p3C_0p4C_file_inventory.csv
[OK] inventory rows = 4
[OK] Loaded Day22A retained trajectories: 4
[OK] Wrote Day22A load sanity summary: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step1_MJ1_0p3C_0p4C_loaded_trajectory_sanity.csv
                          file_name   protocol_label protocol_role                csv_format_refined  n_raw_rows  n_retained_rows  charge_onset_idx_raw charge_onset_timestamp_raw  retained_duration_s  dt_median_s  dt_min_s  dt_max_s  finite_voltage_fraction  finite_current_fraction  U_min_V  U_max_V  I_Q_min_A  I_Q_max_A  I_Q_mean_A
MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv 0.3C+0.4C 0.1tau          DCAC processed_1Hz_aligned_no_metadata       13257            13257                     0                    0:00:01              13256.0          1.0       1.0       1.0                 0.999774                 0.999774 2.845303 4.200315  -0.340000   2.397659    0.884362
 

In [10]:
# Day22A Cell 4 — Event / AC-off detection audit
#
# Purpose:
# - Detect Vmax event time from voltage
# - Detect AC-off / transition support from current waveform
# - Persist event-detection method and AC-off lag audit
#
# Explicitly NOT done here:
# - No Q integration
# - No Segment A/B/D assignment
# - No Δt_raw / Δt_geom / Δt_resid
# - No mechanism verdict

OUT_DAY22A_EVENT_AUDIT = DATA_DIR / "day22A_step2_MJ1_0p3C_0p4C_event_acoff_audit.csv"

# Local Day22A event-detection constants inherited from Day21A contract
DEGLITCH_MIN_CONSECUTIVE_SAMPLES_DAY22 = 3
DEGLITCH_NEAR_VMAX_TOL_V_DAY22 = 0.002

AC_OFF_ENVELOPE_TOLERANCE_COEFFICIENT_DAY22 = 0.05
AC_OFF_PERSISTENCE_N_T_AC_DAY22 = 3

AC_OFF_LOW_FREQUENCY_VOLTAGE_WINDOW_V_DAY22 = [4.195, 4.205]
AC_OFF_LOW_FREQUENCY_MIN_PERSISTENCE_S_DAY22 = 120.0


# =============================================================================
# 4.1 Vmax detection
# =============================================================================

def detect_vmax_event_day22(
    df: pd.DataFrame,
    vmax_v: float = VMAX_V,
    near_tol_v: float = DEGLITCH_NEAR_VMAX_TOL_V_DAY22,
    min_consecutive: int = DEGLITCH_MIN_CONSECUTIVE_SAMPLES_DAY22,
) -> dict[str, object]:
    """
    Priority 3 fallback detection:
    first V >= Vmax with deglitch validation.

    Deglitch:
    - first sample V >= Vmax
    - next min_consecutive samples are near/above Vmax - near_tol_v
    """
    U = df["U_V"].to_numpy(dtype=float)
    t = df["t_s"].to_numpy(dtype=float)

    finite = np.isfinite(U) & np.isfinite(t)
    candidate_idxs = np.where(finite & (U >= vmax_v))[0]

    for idx in candidate_idxs:
        end = min(idx + min_consecutive, len(df))
        if end - idx < min_consecutive:
            continue

        window_U = U[idx:end]
        if np.all(np.isfinite(window_U)) and np.all(window_U >= vmax_v - near_tol_v):
            return {
                "idx_Vmax": int(idx),
                "t_Vmax_detected_s": float(t[idx]),
                "U_at_Vmax_detected_V": float(U[idx]),
                "Vmax_detection_method_used": "priority3_first_V_ge_4p2V_with_deglitch",
                "Vmax_detection_status": "detected",
            }

    if len(candidate_idxs) > 0:
        idx = int(candidate_idxs[0])
        return {
            "idx_Vmax": idx,
            "t_Vmax_detected_s": float(t[idx]),
            "U_at_Vmax_detected_V": float(U[idx]),
            "Vmax_detection_method_used": "priority3_first_V_ge_4p2V_without_deglitch_warning",
            "Vmax_detection_status": "detected_without_deglitch",
        }

    return {
        "idx_Vmax": np.nan,
        "t_Vmax_detected_s": np.nan,
        "U_at_Vmax_detected_V": np.nan,
        "Vmax_detection_method_used": "unresolved_no_V_ge_Vmax",
        "Vmax_detection_status": "unresolved",
    }


# =============================================================================
# 4.2 AC-off detection support from current waveform
# =============================================================================

def ac_off_window_supports_no_sinusoid_day22(
    df: pd.DataFrame,
    idx_candidate: int,
    persistence_s: float,
    ac_amp_A: float,
    voltage_window: Optional[Sequence[float]] = None,
) -> dict[str, object]:
    """
    Check whether the post-candidate current window supports AC disappearance.

    Contract-compatible operational checks:
    - post-candidate record covers required persistence window
    - if voltage_window is provided, candidate voltage lies inside it
    - current does not show a negative AC half-wave after candidate:
      min(I_Q) >= -0.05 * I_AC
    """
    t = df["t_s"].to_numpy(dtype=float)
    U = df["U_V"].to_numpy(dtype=float)
    I = df["I_Q_A"].to_numpy(dtype=float)

    if idx_candidate < 0 or idx_candidate >= len(df):
        return {"ok": False, "reason": "candidate_index_out_of_range"}

    t0 = t[idx_candidate]
    if not np.isfinite(t0):
        return {"ok": False, "reason": "candidate_time_not_finite"}

    if not np.isfinite(persistence_s) or persistence_s <= 0:
        return {"ok": False, "reason": "invalid_persistence_s"}

    if not np.isfinite(ac_amp_A) or ac_amp_A <= 0:
        return {"ok": False, "reason": "invalid_ac_amp_A"}

    if voltage_window is not None:
        lo, hi = float(voltage_window[0]), float(voltage_window[1])
        U0 = U[idx_candidate]
        if not np.isfinite(U0) or not (lo <= U0 <= hi):
            return {
                "ok": False,
                "reason": "candidate_not_in_voltage_window",
                "U_candidate_V": float(U0) if np.isfinite(U0) else np.nan,
            }

    mask = (t >= t0) & (t <= t0 + persistence_s)
    n_window = int(np.sum(mask))

    if n_window < 3:
        return {"ok": False, "reason": "insufficient_samples_in_window"}

    t_end_available = np.nanmax(t)
    if t_end_available < t0 + persistence_s:
        return {
            "ok": False,
            "reason": "post_transition_record_shorter_than_persistence",
            "post_available_s": float(t_end_available - t0),
            "required_persistence_s": float(persistence_s),
        }

    Iw = I[mask]
    finite_i = Iw[np.isfinite(Iw)]

    if len(finite_i) < 3:
        return {"ok": False, "reason": "insufficient_finite_current_samples"}

    tol_A = AC_OFF_ENVELOPE_TOLERANCE_COEFFICIENT_DAY22 * float(ac_amp_A)
    min_i = float(np.nanmin(finite_i))
    max_i = float(np.nanmax(finite_i))
    mean_i = float(np.nanmean(finite_i))

    no_negative_ac_halfwave = min_i >= -tol_A

    return {
        "ok": bool(no_negative_ac_halfwave),
        "reason": "ok" if no_negative_ac_halfwave else "negative_ac_halfwave_detected",
        "n_window": n_window,
        "required_persistence_s": float(persistence_s),
        "tol_A": float(tol_A),
        "I_window_min_A": min_i,
        "I_window_max_A": max_i,
        "I_window_mean_A": mean_i,
        "post_available_s": float(t_end_available - t0),
    }


def detect_acoff_event_day22(
    df: pd.DataFrame,
    inv_row: pd.Series,
    vmax_result: dict[str, object],
) -> dict[str, object]:
    """
    Detect AC-off support for DCAC protocols.

    For pure DC:
    - t_AC_off_detected_s = NaN
    - method = not_applicable_DC_reference

    For DCAC:
    - use Vmax candidate as transition candidate
    - first try default persistence = 3 * T_AC
    - if default persistence cannot be satisfied, try low-frequency fallback
    """
    protocol_role = str(inv_row["protocol_role"])

    if protocol_role == DAY22A_PROTOCOL_ROLE_DC:
        return {
            "idx_AC_off": np.nan,
            "t_AC_off_detected_s": np.nan,
            "AC_off_detection_method_used": "not_applicable_DC_reference",
            "AC_off_detection_status": "not_applicable",
            "AC_off_detection_reason": "pure_DC_reference_has_no_AC_off",
            "AC_off_post_available_s": np.nan,
            "AC_off_required_persistence_s": np.nan,
            "AC_off_window_min_I_A": np.nan,
            "AC_off_window_max_I_A": np.nan,
        }

    idx_vmax = vmax_result.get("idx_Vmax", np.nan)
    if not is_finite_number(idx_vmax):
        return {
            "idx_AC_off": np.nan,
            "t_AC_off_detected_s": np.nan,
            "AC_off_detection_method_used": "unresolved_no_Vmax_candidate",
            "AC_off_detection_status": "unresolved",
            "AC_off_detection_reason": "Vmax_not_detected",
            "AC_off_post_available_s": np.nan,
            "AC_off_required_persistence_s": np.nan,
            "AC_off_window_min_I_A": np.nan,
            "AC_off_window_max_I_A": np.nan,
        }

    idx_vmax = int(idx_vmax)

    ac_amp_A = float(inv_row["AC_C"]) * ONE_C_A
    frequency_hz = float(inv_row["frequency_Hz"])
    T_AC_s = compute_t_ac_s(frequency_hz)

    if not np.isfinite(T_AC_s):
        return {
            "idx_AC_off": np.nan,
            "t_AC_off_detected_s": np.nan,
            "AC_off_detection_method_used": "unresolved_invalid_frequency",
            "AC_off_detection_status": "unresolved",
            "AC_off_detection_reason": "invalid_frequency_Hz",
            "AC_off_post_available_s": np.nan,
            "AC_off_required_persistence_s": np.nan,
            "AC_off_window_min_I_A": np.nan,
            "AC_off_window_max_I_A": np.nan,
        }

    search_last = min(idx_vmax + 60, len(df) - 1)
    candidate_indices = range(idx_vmax, search_last + 1)

    # Default rule: 3 * T_AC
    default_persistence_s = AC_OFF_PERSISTENCE_N_T_AC_DAY22 * T_AC_s

    for idx_candidate in candidate_indices:
        support = ac_off_window_supports_no_sinusoid_day22(
            df=df,
            idx_candidate=idx_candidate,
            persistence_s=default_persistence_s,
            ac_amp_A=ac_amp_A,
            voltage_window=None,
        )
        if support["ok"]:
            t_candidate = float(df["t_s"].iloc[idx_candidate])
            return {
                "idx_AC_off": int(idx_candidate),
                "t_AC_off_detected_s": t_candidate,
                "AC_off_detection_method_used": "priority2_current_envelope_default_3TAC_after_Vmax",
                "AC_off_detection_status": "detected",
                "AC_off_detection_reason": support["reason"],
                "AC_off_post_available_s": support.get("post_available_s", np.nan),
                "AC_off_required_persistence_s": support.get("required_persistence_s", np.nan),
                "AC_off_window_min_I_A": support.get("I_window_min_A", np.nan),
                "AC_off_window_max_I_A": support.get("I_window_max_A", np.nan),
            }

    # Fallback: low-frequency exception
    for idx_candidate in candidate_indices:
        support = ac_off_window_supports_no_sinusoid_day22(
            df=df,
            idx_candidate=idx_candidate,
            persistence_s=AC_OFF_LOW_FREQUENCY_MIN_PERSISTENCE_S_DAY22,
            ac_amp_A=ac_amp_A,
            voltage_window=AC_OFF_LOW_FREQUENCY_VOLTAGE_WINDOW_V_DAY22,
        )
        if support["ok"]:
            t_candidate = float(df["t_s"].iloc[idx_candidate])
            return {
                "idx_AC_off": int(idx_candidate),
                "t_AC_off_detected_s": t_candidate,
                "AC_off_detection_method_used": "priority2_low_frequency_exception_120s_voltage_window",
                "AC_off_detection_status": "detected",
                "AC_off_detection_reason": support["reason"],
                "AC_off_post_available_s": support.get("post_available_s", np.nan),
                "AC_off_required_persistence_s": support.get("required_persistence_s", np.nan),
                "AC_off_window_min_I_A": support.get("I_window_min_A", np.nan),
                "AC_off_window_max_I_A": support.get("I_window_max_A", np.nan),
            }

    return {
        "idx_AC_off": np.nan,
        "t_AC_off_detected_s": np.nan,
        "AC_off_detection_method_used": "unresolved_current_envelope_detection_failed",
        "AC_off_detection_status": "unresolved",
        "AC_off_detection_reason": "no_candidate_satisfied_default_or_low_frequency_exception",
        "AC_off_post_available_s": np.nan,
        "AC_off_required_persistence_s": default_persistence_s,
        "AC_off_window_min_I_A": np.nan,
        "AC_off_window_max_I_A": np.nan,
    }


# =============================================================================
# 4.3 Run event audit
# =============================================================================

event_rows = []

for _, inv_row in df_day22_inventory.iterrows():
    file_name = inv_row["file_name"]
    df_traj = TRAJ22[file_name]

    vmax_result = detect_vmax_event_day22(df_traj)
    acoff_result = detect_acoff_event_day22(df_traj, inv_row, vmax_result)

    t_vmax = vmax_result["t_Vmax_detected_s"]
    t_acoff = acoff_result["t_AC_off_detected_s"]

    if is_finite_number(t_vmax) and is_finite_number(t_acoff):
        ac_off_lag_s = float(t_acoff) - float(t_vmax)
    else:
        ac_off_lag_s = np.nan

    row = {
        "file_name": file_name,
        "protocol_label": inv_row["protocol_label"],
        "protocol_role": inv_row["protocol_role"],
        "DC_C": inv_row["DC_C"],
        "AC_C": inv_row["AC_C"],
        "m_tau": inv_row["m_tau"],
        "frequency_Hz": inv_row["frequency_Hz"],
        "csv_format_refined": inv_row["csv_format_refined"],

        **vmax_result,
        **acoff_result,

        "AC_off_lag_s": ac_off_lag_s,
    }

    event_rows.append(row)

df_day22_event_audit = pd.DataFrame(event_rows)

def event_framework_status_day22(row: pd.Series) -> str:
    if row["protocol_role"] == DAY22A_PROTOCOL_ROLE_DC:
        return "event_audit_ok_DC_reference"

    if row["Vmax_detection_status"] not in ["detected", "detected_without_deglitch"]:
        return "Vmax_unresolved"

    if row["AC_off_detection_status"] != "detected":
        return "AC_off_unresolved"

    if is_finite_number(row["AC_off_lag_s"]) and float(row["AC_off_lag_s"]) < 0:
        return SEGMENT_FRAMEWORK_AC_OFF_PRECEDES_VMAX

    return "event_audit_ok_DCAC"


df_day22_event_audit["event_framework_status"] = df_day22_event_audit.apply(
    event_framework_status_day22,
    axis=1,
)

OUT_DAY22A_EVENT_AUDIT = DATA_DIR / "day22A_step2_MJ1_0p3C_0p4C_event_acoff_audit.csv"
df_day22_event_audit.to_csv(OUT_DAY22A_EVENT_AUDIT, index=False)

print(f"[OK] Wrote Day22A event / AC-off audit: {OUT_DAY22A_EVENT_AUDIT}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "csv_format_refined",
    "t_Vmax_detected_s",
    "U_at_Vmax_detected_V",
    "Vmax_detection_method_used",
    "t_AC_off_detected_s",
    "AC_off_detection_method_used",
    "AC_off_lag_s",
    "AC_off_required_persistence_s",
    "AC_off_post_available_s",
    "AC_off_window_min_I_A",
    "AC_off_window_max_I_A",
    "event_framework_status",
]

print(df_day22_event_audit[display_cols].to_string(index=False))

# =============================================================================
# 4.4 Hard guards
# =============================================================================

bad_event = df_day22_event_audit[
    ~df_day22_event_audit["event_framework_status"].isin([
        "event_audit_ok_DC_reference",
        "event_audit_ok_DCAC",
    ])
]

if len(bad_event) > 0:
    print("[warning] Day22A event audit has unresolved / non-OK rows:")
    print(bad_event[display_cols].to_string(index=False))
    raise ValueError("Day22A event / AC-off audit did not pass for all required trajectories.")

print("[OK] Cell 4 Day22A event / AC-off audit passed.")
print("[OK] No Q integration, segment assignment, Δt computation, or verdict performed.")

[OK] Wrote Day22A event / AC-off audit: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step2_MJ1_0p3C_0p4C_event_acoff_audit.csv
                          file_name   protocol_label protocol_role                csv_format_refined  t_Vmax_detected_s  U_at_Vmax_detected_V              Vmax_detection_method_used  t_AC_off_detected_s                       AC_off_detection_method_used  AC_off_lag_s  AC_off_required_persistence_s  AC_off_post_available_s  AC_off_window_min_I_A  AC_off_window_max_I_A      event_framework_status
MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv 0.3C+0.4C 0.1tau          DCAC processed_1Hz_aligned_no_metadata             9506.0              4.200284 priority3_first_V_ge_4p2V_with_deglitch               9506.0 priority2_current_envelope_default_3TAC_after_Vmax           0.0                      20.923007                   3750.0               1.904491               2.397659         event_audit_ok_DCAC
 MJ1_0p3C_0p4C_10tau_NGU201_raw.csv  0.3C+0.4C 10tau          DCAC pro

In [11]:
# Day22A Cell 5 — Strict-net Q integration and final-Q consistency audit
#
# Purpose:
# - Compute strict-net Q_net(t) for each retained Day22A trajectory
# - Preserve signed current, no rectification, no cummax
# - Audit local non-monotonicity caused by DCAC negative-current intervals
# - Compute final-Q consistency against the 0.3C DC reference
#
# Explicitly NOT done here:
# - No event charge extraction
# - No Segment A/B/D assignment
# - No Δt_raw / Δt_geom / Δt_resid
# - No mechanism verdict

OUT_DAY22A_Q_SUMMARY = DATA_DIR / "day22A_step3_MJ1_0p3C_0p4C_Q_integration_summary.csv"
OUT_DAY22A_FINALQ_PAIR_AUDIT = DATA_DIR / "day22A_step3_MJ1_0p3C_0p4C_finalQ_pair_audit.csv"


# =============================================================================
# 5.1 Strict-net signed trapezoidal integration
# =============================================================================

def integrate_strict_net_Q_Ah_day22(
    t_s: np.ndarray,
    i_q_A: np.ndarray,
) -> np.ndarray:
    """
    Strict-net signed trapezoidal integration.

    Rules:
    - Use signed charge-positive current I_Q_A.
    - Do not rectify.
    - Do not apply cummax.
    - Preserve local decreases in Q caused by negative-current intervals.
    - Q_net(t=0) = 0.
    """
    t = np.asarray(t_s, dtype=float)
    i = np.asarray(i_q_A, dtype=float)

    if len(t) != len(i):
        raise ValueError("time and current arrays must have the same length")

    q = np.zeros(len(t), dtype=float)
    if len(t) == 0:
        return q

    for k in range(1, len(t)):
        if (
            np.isfinite(t[k])
            and np.isfinite(t[k - 1])
            and np.isfinite(i[k])
            and np.isfinite(i[k - 1])
        ):
            dt_h = (t[k] - t[k - 1]) / 3600.0
            if dt_h > 0:
                q[k] = q[k - 1] + 0.5 * (i[k] + i[k - 1]) * dt_h
            else:
                q[k] = q[k - 1]
        else:
            q[k] = q[k - 1]

    return q


# =============================================================================
# 5.2 Apply integration
# =============================================================================

q_summary_rows = []

for file_name, df_traj in TRAJ22.items():
    t_s = df_traj["t_s"].to_numpy(dtype=float)
    i_q = df_traj["I_Q_A"].to_numpy(dtype=float)

    q_net_Ah = integrate_strict_net_Q_Ah_day22(t_s, i_q)

    TRAJ22[file_name] = df_traj.copy()
    TRAJ22[file_name]["Q_net_Ah"] = q_net_Ah

    dq = np.diff(q_net_Ah)
    finite_dq = dq[np.isfinite(dq)]

    n_q_decrease = int(np.sum(finite_dq < -1e-12)) if len(finite_dq) else 0
    q_decrease_fraction = (
        float(n_q_decrease / len(finite_dq)) if len(finite_dq) else np.nan
    )

    q_final = float(q_net_Ah[-1]) if len(q_net_Ah) else np.nan
    q_max = float(np.nanmax(q_net_Ah)) if len(q_net_Ah) else np.nan
    q_min = float(np.nanmin(q_net_Ah)) if len(q_net_Ah) else np.nan

    inv_row = df_day22_inventory.loc[df_day22_inventory["file_name"] == file_name].iloc[0]

    q_summary_rows.append({
        "file_name": file_name,
        "protocol_label": inv_row["protocol_label"],
        "protocol_role": inv_row["protocol_role"],
        "DC_C": inv_row["DC_C"],
        "AC_C": inv_row["AC_C"],
        "m_tau": inv_row["m_tau"],
        "frequency_Hz": inv_row["frequency_Hz"],
        "csv_format_refined": inv_row["csv_format_refined"],
        "n_retained_rows": len(TRAJ22[file_name]),
        "t_final_s": float(TRAJ22[file_name]["t_s"].iloc[-1]),
        "Q_final_Ah": q_final,
        "Q_final_mAh": q_final * 1000.0 if np.isfinite(q_final) else np.nan,
        "Q_max_Ah": q_max,
        "Q_min_Ah": q_min,
        "Q_decrease_count": n_q_decrease,
        "Q_decrease_fraction": q_decrease_fraction,
        "I_Q_min_A": float(np.nanmin(i_q)),
        "I_Q_max_A": float(np.nanmax(i_q)),
        "I_Q_mean_A": float(np.nanmean(i_q)),
    })

df_day22_q_summary = pd.DataFrame(q_summary_rows)
df_day22_q_summary.to_csv(OUT_DAY22A_Q_SUMMARY, index=False)

print(f"[OK] Wrote Day22A Q integration summary: {OUT_DAY22A_Q_SUMMARY}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "csv_format_refined",
    "t_final_s",
    "Q_final_Ah",
    "Q_final_mAh",
    "Q_max_Ah",
    "Q_decrease_count",
    "Q_decrease_fraction",
    "I_Q_min_A",
    "I_Q_max_A",
    "I_Q_mean_A",
]
print(df_day22_q_summary[display_cols].to_string(index=False))


# =============================================================================
# 5.3 Final-Q consistency audit
# =============================================================================

def compute_common_anchors_day22(
    q_final_dc_ah: float,
    q_final_dcac_ah: float,
    q_nom_ah: float = Q_NOM_AH,
) -> dict[str, float]:
    if not is_finite_number(q_final_dc_ah) or not is_finite_number(q_final_dcac_ah):
        return {
            "Q_common_final_Ah": np.nan,
            "Q80_common_Ah": np.nan,
            "Q90_common_Ah": np.nan,
            "Q80_common_fraction_of_Q_nom": np.nan,
            "Q90_common_fraction_of_Q_nom": np.nan,
        }

    q_common_final = min(float(q_final_dc_ah), float(q_final_dcac_ah))
    q80_common = Q80_NOMINAL_FRACTION_OF_Q_NOM * q_common_final
    q90_common = Q90_NOMINAL_FRACTION_OF_Q_NOM * q_common_final

    return {
        "Q_common_final_Ah": q_common_final,
        "Q80_common_Ah": q80_common,
        "Q90_common_Ah": q90_common,
        "Q80_common_fraction_of_Q_nom": q80_common / q_nom_ah,
        "Q90_common_fraction_of_Q_nom": q90_common / q_nom_ah,
    }


def final_q_status_day22(q_final_dc_ah: float, q_final_dcac_ah: float) -> str:
    if not is_finite_number(q_final_dc_ah) or not is_finite_number(q_final_dcac_ah):
        return FINAL_Q_UNRESOLVED

    diff = abs(float(q_final_dc_ah) - float(q_final_dcac_ah))
    return FINAL_Q_CONSISTENT if diff <= FINAL_Q_DIFF_THRESHOLD_AH else FINAL_Q_MISMATCH_WARNING


dc_rows = df_day22_q_summary[df_day22_q_summary["protocol_role"] == DAY22A_PROTOCOL_ROLE_DC]
if len(dc_rows) != 1:
    raise ValueError(f"Expected exactly one Day22A DC reference, found {len(dc_rows)}")

dc_ref = dc_rows.iloc[0]
q_final_dc = float(dc_ref["Q_final_Ah"])

pair_rows = []

for _, row in df_day22_q_summary.iterrows():
    if row["protocol_role"] != DAY22A_PROTOCOL_ROLE_DCAC:
        continue

    q_final_dcac = float(row["Q_final_Ah"])
    diff = q_final_dc - q_final_dcac
    abs_diff = abs(diff)

    common = compute_common_anchors_day22(
        q_final_dc_ah=q_final_dc,
        q_final_dcac_ah=q_final_dcac,
        q_nom_ah=Q_NOM_AH,
    )

    pair_rows.append({
        "protocol_pair": f"{dc_ref['protocol_label']} vs {row['protocol_label']}",
        "protocol_label_DC": dc_ref["protocol_label"],
        "protocol_label_DCAC": row["protocol_label"],
        "Q_nom_Ah": Q_NOM_AH,
        "Q_final_DC_Ah": q_final_dc,
        "Q_final_DCAC_Ah": q_final_dcac,
        "Q_final_diff_Ah": diff,
        "Q_final_abs_diff_Ah": abs_diff,
        "Q_final_diff_mAh": diff * 1000.0,
        "Q_final_abs_diff_mAh": abs_diff * 1000.0,
        "Q_final_diff_status": final_q_status_day22(q_final_dc, q_final_dcac),
        "Q80_nominal_Ah": Q80_NOMINAL_AH,
        "Q90_nominal_Ah": Q90_NOMINAL_AH,
        "Q80_nominal_fraction_of_Q_nom": Q80_NOMINAL_FRACTION_OF_Q_NOM,
        "Q90_nominal_fraction_of_Q_nom": Q90_NOMINAL_FRACTION_OF_Q_NOM,
        "Q80_common_Ah": common["Q80_common_Ah"],
        "Q90_common_Ah": common["Q90_common_Ah"],
        "Q80_common_fraction_of_Q_nom": common["Q80_common_fraction_of_Q_nom"],
        "Q90_common_fraction_of_Q_nom": common["Q90_common_fraction_of_Q_nom"],
        "Q_common_final_Ah_internal": common["Q_common_final_Ah"],
    })

df_day22_finalq_pairs = pd.DataFrame(pair_rows)
df_day22_finalq_pairs.to_csv(OUT_DAY22A_FINALQ_PAIR_AUDIT, index=False)

print(f"[OK] Wrote Day22A final-Q pair audit: {OUT_DAY22A_FINALQ_PAIR_AUDIT}")

display_cols = [
    "protocol_pair",
    "Q_final_DC_Ah",
    "Q_final_DCAC_Ah",
    "Q_final_diff_mAh",
    "Q_final_abs_diff_mAh",
    "Q_final_diff_status",
    "Q80_common_Ah",
    "Q90_common_Ah",
    "Q80_common_fraction_of_Q_nom",
    "Q90_common_fraction_of_Q_nom",
]
print(df_day22_finalq_pairs[display_cols].to_string(index=False))


# =============================================================================
# 5.4 Hard guards
# =============================================================================

if not np.isfinite(q_final_dc) or q_final_dc <= 0:
    raise ValueError("Day22A DC reference final Q is invalid.")

for _, row in df_day22_q_summary.iterrows():
    if not np.isfinite(row["Q_final_Ah"]) or row["Q_final_Ah"] <= 0:
        raise ValueError(f"{row['file_name']}: invalid Q_final_Ah.")

n_mismatch = int((df_day22_finalq_pairs["Q_final_diff_status"] == FINAL_Q_MISMATCH_WARNING).sum())
if n_mismatch > 0:
    print(f"[warning] Final-Q mismatch warning in {n_mismatch} Day22A DCAC pair(s).")
    print("          Common anchors remain valid as shared reachable absolute-Q targets.")
    print("          Later verdicts using common anchors must carry asymmetric_final_Q caveat.")

print("[OK] Cell 5 Day22A strict-net Q integration and final-Q audit completed.")
print("[OK] No event charge extraction, segment assignment, Δt computation, or verdict performed.")

[OK] Wrote Day22A Q integration summary: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step3_MJ1_0p3C_0p4C_Q_integration_summary.csv
                          file_name   protocol_label protocol_role                csv_format_refined  t_final_s  Q_final_Ah  Q_final_mAh  Q_max_Ah  Q_decrease_count  Q_decrease_fraction  I_Q_min_A  I_Q_max_A  I_Q_mean_A
MJ1_0p3C_0p4C_0p1tau_NGU201_raw.csv 0.3C+0.4C 0.1tau          DCAC processed_1Hz_aligned_no_metadata    13256.0    3.255150  3255.149661  3.255150              1766             0.133223  -0.340000   2.397659    0.884362
 MJ1_0p3C_0p4C_10tau_NGU201_raw.csv  0.3C+0.4C 10tau          DCAC processed_1Hz_aligned_no_metadata    13030.0    3.252939  3252.939148  3.252939              2093             0.160629  -0.340000   2.474556    0.899167
  MJ1_0p3C_0p4C_1tau_NGU201_raw.csv   0.3C+0.4C 1tau          DCAC                    NGU201_LOG_raw    13288.2    3.259023  3259.023139  3.259023              2168             0.163192  -0.340066   2.

In [12]:
# Day22A Cell 5A — Experimental resolution-floor diagnostic
#
# Purpose:
# - Estimate a Day22A self-consistency lower-bound floor from the 0.3C DC reference
# - Use even/odd interleaved subsets of the same DC trajectory
# - Quantify first-passage sensitivity due to sampling / interpolation / integration
#
# Explicitly NOT done here:
# - No mechanism verdict
# - No Segment A/B/D assignment
# - No residual threshold redefinition
# - No claim of repeat-based experimental noise floor

OUT_DAY22A_RESOLUTION_LONG = DATA_DIR / "day22A_step3A_MJ1_0p3C_0p4C_resolution_floor_long.csv"
OUT_DAY22A_RESOLUTION_SUMMARY = DATA_DIR / "day22A_step3A_MJ1_0p3C_0p4C_resolution_floor_summary.csv"


# =============================================================================
# 5A.1 Helper functions
# =============================================================================

def first_passage_time_from_Q_day22(
    q_target_Ah: float,
    t_s: np.ndarray,
    q_Ah: np.ndarray,
) -> float:
    """
    First-passage time: first t where Q(t) >= q_target.
    No monotonic correction is applied.
    """
    if not is_finite_number(q_target_Ah):
        return np.nan

    t = np.asarray(t_s, dtype=float)
    q = np.asarray(q_Ah, dtype=float)

    finite = np.isfinite(t) & np.isfinite(q)
    if finite.sum() < 2:
        return np.nan

    t_f = t[finite]
    q_f = q[finite]

    hit = np.where(q_f >= float(q_target_Ah))[0]
    if len(hit) == 0:
        return np.nan

    idx = int(hit[0])

    if idx == 0:
        return float(t_f[idx])

    q0, q1 = q_f[idx - 1], q_f[idx]
    t0, t1 = t_f[idx - 1], t_f[idx]

    if not np.isfinite(q0) or not np.isfinite(q1) or q1 == q0:
        return float(t1)

    frac = (float(q_target_Ah) - q0) / (q1 - q0)
    frac = float(np.clip(frac, 0.0, 1.0))
    return float(t0 + frac * (t1 - t0))


def make_fixed_Q_grid_day22(q_lo_Ah: float, q_hi_Ah: float, step_Ah: float) -> np.ndarray:
    """
    Fixed Q-grid from first grid value >= q_lo to last grid value <= q_hi.
    """
    if not all(is_finite_number(x) for x in [q_lo_Ah, q_hi_Ah, step_Ah]):
        return np.array([], dtype=float)

    if q_hi_Ah < q_lo_Ah or step_Ah <= 0:
        return np.array([], dtype=float)

    start = np.ceil(float(q_lo_Ah) / step_Ah) * step_Ah
    stop = np.floor(float(q_hi_Ah) / step_Ah) * step_Ah

    if stop < start:
        return np.array([], dtype=float)

    n = int(round((stop - start) / step_Ah)) + 1
    return start + step_Ah * np.arange(n)


def integrate_subset_day22(df: pd.DataFrame, subset_name: str, selector: np.ndarray) -> pd.DataFrame:
    """
    Build a self-consistency subset from one trajectory.

    Important:
    - original t_s is preserved;
    - Q is re-integrated only on the selected subset;
    - this is a lower-bound diagnostic for sampling/interpolation sensitivity,
      not a repeatability estimate.
    """
    out = df.loc[selector].copy().reset_index(drop=True)

    t = out["t_s"].to_numpy(dtype=float)
    i = out["I_Q_A"].to_numpy(dtype=float)

    out["Q_self_Ah"] = integrate_strict_net_Q_Ah_day22(t, i)
    out["self_subset"] = subset_name

    return out


# =============================================================================
# 5A.2 Build DC self-consistency subsets
# =============================================================================

dc_rows = df_day22_inventory[df_day22_inventory["candidate_for_DC_reference"].map(parse_bool_strict_day22)]
if len(dc_rows) != 1:
    raise ValueError(f"Expected exactly one Day22A DC reference, found {len(dc_rows)}")

dc_file = dc_rows.iloc[0]["file_name"]
dc_traj = TRAJ22[dc_file].copy()

if "Q_net_Ah" not in dc_traj.columns:
    raise ValueError("DC trajectory does not contain Q_net_Ah. Run Day22A Cell 5 first.")

idx = np.arange(len(dc_traj))

df_dc_full = dc_traj.copy()
df_dc_full["Q_self_Ah"] = df_dc_full["Q_net_Ah"]
df_dc_full["self_subset"] = "full"

df_dc_even = integrate_subset_day22(dc_traj, "even_rows", idx % 2 == 0)
df_dc_odd = integrate_subset_day22(dc_traj, "odd_rows", idx % 2 == 1)

q_hi_common = min(
    float(df_dc_full["Q_self_Ah"].iloc[-1]),
    float(df_dc_even["Q_self_Ah"].iloc[-1]),
    float(df_dc_odd["Q_self_Ah"].iloc[-1]),
)

# Leave one grid step margin at the top to avoid endpoint instability.
q_lo = SEGMENT_A_Q_LO_AH
q_hi = q_hi_common - Q_GRID_STEP_AH

q_grid = make_fixed_Q_grid_day22(q_lo, q_hi, Q_GRID_STEP_AH)

if len(q_grid) < Q_GRID_MIN_COUNT_SEGMENT_A:
    raise ValueError(
        f"Resolution-floor Q-grid too short: n={len(q_grid)}. "
        f"Need >= {Q_GRID_MIN_COUNT_SEGMENT_A}."
    )

print(f"[OK] DC self-consistency reference file: {dc_file}")
print(f"[OK] Q-grid for self-consistency: n={len(q_grid)}, lo={q_grid[0]:.3f} Ah, hi={q_grid[-1]:.3f} Ah")


# =============================================================================
# 5A.3 Compute self-Δt diagnostics
# =============================================================================

resolution_rows = []

for q in q_grid:
    t_full = first_passage_time_from_Q_day22(
        q,
        df_dc_full["t_s"].to_numpy(dtype=float),
        df_dc_full["Q_self_Ah"].to_numpy(dtype=float),
    )

    t_even = first_passage_time_from_Q_day22(
        q,
        df_dc_even["t_s"].to_numpy(dtype=float),
        df_dc_even["Q_self_Ah"].to_numpy(dtype=float),
    )

    t_odd = first_passage_time_from_Q_day22(
        q,
        df_dc_odd["t_s"].to_numpy(dtype=float),
        df_dc_odd["Q_self_Ah"].to_numpy(dtype=float),
    )

    resolution_rows.append({
        "Q_Ah": float(q),
        "t_full_s": t_full,
        "t_even_s": t_even,
        "t_odd_s": t_odd,
        "dt_even_minus_odd_s": (
            t_even - t_odd if is_finite_number(t_even) and is_finite_number(t_odd) else np.nan
        ),
        "dt_full_minus_even_s": (
            t_full - t_even if is_finite_number(t_full) and is_finite_number(t_even) else np.nan
        ),
        "dt_full_minus_odd_s": (
            t_full - t_odd if is_finite_number(t_full) and is_finite_number(t_odd) else np.nan
        ),
    })

df_resolution_long = pd.DataFrame(resolution_rows)

def safe_abs_p95(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.percentile(np.abs(arr), 95)) if len(arr) else np.nan

def safe_abs_max(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.max(np.abs(arr))) if len(arr) else np.nan

def safe_mean(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.mean(arr)) if len(arr) else np.nan

def safe_median(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.median(arr)) if len(arr) else np.nan

p95_even_odd = safe_abs_p95(df_resolution_long["dt_even_minus_odd_s"])
p95_full_even = safe_abs_p95(df_resolution_long["dt_full_minus_even_s"])
p95_full_odd = safe_abs_p95(df_resolution_long["dt_full_minus_odd_s"])

max_even_odd = safe_abs_max(df_resolution_long["dt_even_minus_odd_s"])
max_full_even = safe_abs_max(df_resolution_long["dt_full_minus_even_s"])
max_full_odd = safe_abs_max(df_resolution_long["dt_full_minus_odd_s"])

audit_resolution_p95_s = np.nanmax([p95_even_odd, p95_full_even, p95_full_odd])
audit_resolution_max_s = np.nanmax([max_even_odd, max_full_even, max_full_odd])

summary = {
    "source": "0.3C_DC_self_consistency_even_odd_split",
    "dc_reference_file": dc_file,
    "n_Q_grid": int(len(q_grid)),
    "Q_lo_Ah": float(q_grid[0]),
    "Q_hi_Ah": float(q_grid[-1]),
    "Q_grid_step_Ah": Q_GRID_STEP_AH,

    "dt_even_minus_odd_mean_s": safe_mean(df_resolution_long["dt_even_minus_odd_s"]),
    "dt_even_minus_odd_median_s": safe_median(df_resolution_long["dt_even_minus_odd_s"]),
    "dt_even_minus_odd_p95_abs_s": p95_even_odd,
    "dt_even_minus_odd_max_abs_s": max_even_odd,

    "dt_full_minus_even_p95_abs_s": p95_full_even,
    "dt_full_minus_even_max_abs_s": max_full_even,

    "dt_full_minus_odd_p95_abs_s": p95_full_odd,
    "dt_full_minus_odd_max_abs_s": max_full_odd,

    "day22A_self_consistency_resolution_p95_s": audit_resolution_p95_s,
    "day22A_self_consistency_resolution_max_s": audit_resolution_max_s,

    "repeat_based_noise_floor_available": False,
    "floor_scope": "lower_bound_for_sampling_interpolation_first_passage_resolution_not_repeatability",
    "formal_disappearance_claim_allowed": False,
}

df_resolution_summary = pd.DataFrame([summary])

df_resolution_long.to_csv(OUT_DAY22A_RESOLUTION_LONG, index=False)
df_resolution_summary.to_csv(OUT_DAY22A_RESOLUTION_SUMMARY, index=False)

print(f"[OK] Wrote Day22A resolution-floor long table: {OUT_DAY22A_RESOLUTION_LONG}")
print(f"[OK] Wrote Day22A resolution-floor summary: {OUT_DAY22A_RESOLUTION_SUMMARY}")
print(df_resolution_summary.to_string(index=False))


# =============================================================================
# 5A.4 Hard guards and interpretation reminder
# =============================================================================

if not np.isfinite(audit_resolution_p95_s):
    raise ValueError("Day22A self-consistency p95 resolution estimate is not finite.")

if audit_resolution_p95_s <= 0:
    raise ValueError("Day22A self-consistency p95 resolution estimate is non-positive.")

print("[OK] Cell 5A Day22A experimental resolution-floor diagnostic completed.")
print("[OK] This is a lower-bound audit-resolution estimate, not a repeat-based noise floor.")
print("[OK] Day22A must not claim strict effect disappearance without repeat-based evidence.")

[OK] DC self-consistency reference file: MJ1_0p3C_DC_NGU201_raw.csv
[OK] Q-grid for self-consistency: n=320, lo=0.050 Ah, hi=3.240 Ah
[OK] Wrote Day22A resolution-floor long table: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step3A_MJ1_0p3C_0p4C_resolution_floor_long.csv
[OK] Wrote Day22A resolution-floor summary: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step3A_MJ1_0p3C_0p4C_resolution_floor_summary.csv
                                 source          dc_reference_file  n_Q_grid  Q_lo_Ah  Q_hi_Ah  Q_grid_step_Ah  dt_even_minus_odd_mean_s  dt_even_minus_odd_median_s  dt_even_minus_odd_p95_abs_s  dt_even_minus_odd_max_abs_s  dt_full_minus_even_p95_abs_s  dt_full_minus_even_max_abs_s  dt_full_minus_odd_p95_abs_s  dt_full_minus_odd_max_abs_s  day22A_self_consistency_resolution_p95_s  day22A_self_consistency_resolution_max_s  repeat_based_noise_floor_available                                                                       floor_scope  formal_disappearance_claim_all

In [13]:
# Day22A Cell 6 — Event charge extraction and Q-anchor segment assignment
#
# Purpose:
# - Extract Q_net at Vmax and Segment-B start events
# - Compute Q_Vmax_DC_Ah, Q_Vmax_DCAC_Ah, Q_segmentB_start_Ah
# - Assign Q80/Q90 nominal/common anchors to A/B/D/outside
#
# Explicitly NOT done here:
# - No Δt_raw computation
# - No Δt_geom computation
# - No Δt_resid computation
# - No mechanism verdict

OUT_DAY22A_SEGMENT_ASSIGNMENT = DATA_DIR / "day22A_step4_MJ1_0p3C_0p4C_segment_assignment.csv"

if not OUT_DAY22A_EVENT_AUDIT.exists():
    raise FileNotFoundError(
        f"Day22A event audit missing: {OUT_DAY22A_EVENT_AUDIT}\n"
        "Run Day22A Cell 4 first."
    )

if not OUT_DAY22A_FINALQ_PAIR_AUDIT.exists():
    raise FileNotFoundError(
        f"Day22A final-Q pair audit missing: {OUT_DAY22A_FINALQ_PAIR_AUDIT}\n"
        "Run Day22A Cell 5 first."
    )

df_day22_event_audit = pd.read_csv(OUT_DAY22A_EVENT_AUDIT)
df_day22_finalq_pairs = pd.read_csv(OUT_DAY22A_FINALQ_PAIR_AUDIT)

print(f"[OK] Loaded Day22A event audit: {OUT_DAY22A_EVENT_AUDIT}")
print(f"[OK] Loaded Day22A final-Q pair audit: {OUT_DAY22A_FINALQ_PAIR_AUDIT}")


# =============================================================================
# 6.1 Helpers
# =============================================================================

def interp_q_at_time_day22(df_traj: pd.DataFrame, t_target_s: float) -> float:
    """
    Interpolate Q_net_Ah at a detected event time.

    This maps event times to Q coordinates.
    It is not first-passage Δt analysis.
    """
    if not is_finite_number(t_target_s):
        return np.nan

    if "Q_net_Ah" not in df_traj.columns:
        raise ValueError("Trajectory does not contain Q_net_Ah. Run Day22A Cell 5 first.")

    t = df_traj["t_s"].to_numpy(dtype=float)
    q = df_traj["Q_net_Ah"].to_numpy(dtype=float)

    finite = np.isfinite(t) & np.isfinite(q)
    if finite.sum() < 2:
        return np.nan

    t_f = t[finite]
    q_f = q[finite]

    if t_target_s < t_f[0] or t_target_s > t_f[-1]:
        return np.nan

    return float(np.interp(float(t_target_s), t_f, q_f))


def get_day22_event_row(file_name: str) -> pd.Series:
    rows = df_day22_event_audit[df_day22_event_audit["file_name"] == file_name]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one Day22A event row for {file_name}, found {len(rows)}")
    return rows.iloc[0]


def get_day22_inventory_row(file_name: str) -> pd.Series:
    rows = df_day22_inventory[df_day22_inventory["file_name"] == file_name]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one Day22A inventory row for {file_name}, found {len(rows)}")
    return rows.iloc[0]


def get_day22_dc_reference_file_name() -> str:
    rows = df_day22_inventory[df_day22_inventory["candidate_for_DC_reference"].map(parse_bool_strict_day22)]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one Day22A DC reference, found {len(rows)}")
    return str(rows.iloc[0]["file_name"])


def q_boundary_ordering_status_day22(
    q_segmentB_start_ah: float,
    q_vmax_dc_ah: float,
    tol_ah: float = Q_SEGMENTB_DEGENERATE_TOLERANCE_AH,
) -> str:
    """
    Expected ordering:
    Q_segmentB_start_Ah < Q_Vmax_DC_Ah

    Equality within 1 mAh is classified as Segment-B degenerate.
    """
    if not is_finite_number(q_segmentB_start_ah) or not is_finite_number(q_vmax_dc_ah):
        return ORDERING_UNRESOLVED

    q_b = float(q_segmentB_start_ah)
    q_dc = float(q_vmax_dc_ah)

    if q_b < q_dc - tol_ah:
        return ORDERING_EXPECTED

    if abs(q_b - q_dc) <= tol_ah:
        return ORDERING_DEGENERATE

    return ORDERING_VIOLATED


def segment_framework_status_from_ordering_day22(q_ordering_status: str) -> str:
    if q_ordering_status == ORDERING_EXPECTED:
        return SEGMENT_FRAMEWORK_OK
    if q_ordering_status == ORDERING_DEGENERATE:
        return SEGMENT_FRAMEWORK_SEGMENT_B_DEGENERATE
    if q_ordering_status == ORDERING_VIOLATED:
        return SEGMENT_FRAMEWORK_ORDERING_VIOLATED
    return SEGMENT_FRAMEWORK_UNRESOLVED


def assign_segment_by_q_day22(
    q_ah: float,
    q_segmentB_start_ah: float,
    q_vmax_dc_ah: float,
    q_final_dc_ah: float,
    q_final_dcac_ah: float,
    eps_ah: float = 1e-9,
) -> str:
    """
    Assign target Q to Segment A/B/D/outside.

    Rules:
    - outside: Q > min(Q_final_DC, Q_final_DCAC)
    - A: Q <= Q_segmentB_start
    - B: Q_segmentB_start < Q <= Q_Vmax_DC
    - D: Q > Q_Vmax_DC
    """
    required = [
        q_ah,
        q_segmentB_start_ah,
        q_vmax_dc_ah,
        q_final_dc_ah,
        q_final_dcac_ah,
    ]
    if not all(is_finite_number(x) for x in required):
        return SEGMENT_UNRESOLVED

    q = float(q_ah)
    q_b = float(q_segmentB_start_ah)
    q_vdc = float(q_vmax_dc_ah)
    q_common_final = min(float(q_final_dc_ah), float(q_final_dcac_ah))

    if q > q_common_final + eps_ah:
        return SEGMENT_OUTSIDE

    if q <= q_b + eps_ah:
        return SEGMENT_A

    if q <= q_vdc + eps_ah:
        return SEGMENT_B

    return SEGMENT_D


def segment_A_grid_count_day22(
    segment_A_Q_lo_Ah: float,
    segment_A_Q_hi_Ah: float,
    q_grid_step_Ah: float = Q_GRID_STEP_AH,
) -> int:
    if not all(is_finite_number(x) for x in [segment_A_Q_lo_Ah, segment_A_Q_hi_Ah, q_grid_step_Ah]):
        return 0

    q_lo = float(segment_A_Q_lo_Ah)
    q_hi = float(segment_A_Q_hi_Ah)
    step = float(q_grid_step_Ah)

    if q_hi < q_lo or step <= 0:
        return 0

    return int(np.floor((q_hi - q_lo) / step)) + 1


# =============================================================================
# 6.2 Extract DC reference Vmax charge
# =============================================================================

dc_file_day22 = get_day22_dc_reference_file_name()
dc_inv_day22 = get_day22_inventory_row(dc_file_day22)
dc_event_day22 = get_day22_event_row(dc_file_day22)
dc_traj_day22 = TRAJ22[dc_file_day22]

t_vmax_dc_s = float(dc_event_day22["t_Vmax_detected_s"])
q_vmax_dc_ah = interp_q_at_time_day22(dc_traj_day22, t_vmax_dc_s)

if not is_finite_number(q_vmax_dc_ah):
    raise ValueError("Could not extract Day22A Q_Vmax_DC_Ah.")

print(f"[OK] Day22A DC reference file = {dc_file_day22}")
print(f"[OK] t_Vmax_DC_s = {t_vmax_dc_s:.3f}")
print(f"[OK] Q_Vmax_DC_Ah = {q_vmax_dc_ah:.6f}")


# =============================================================================
# 6.3 Pairwise DC-vs-DCAC segment assignment
# =============================================================================

dc_q_summary_day22 = df_day22_q_summary[df_day22_q_summary["file_name"] == dc_file_day22].iloc[0]
q_final_dc = float(dc_q_summary_day22["Q_final_Ah"])

segment_rows = []

for _, inv_row in df_day22_inventory.iterrows():
    if not parse_bool_strict_day22(inv_row["candidate_for_DCAC"]):
        continue

    dcac_file = str(inv_row["file_name"])
    dcac_event = get_day22_event_row(dcac_file)
    dcac_traj = TRAJ22[dcac_file]

    q_summary_dcac = df_day22_q_summary[df_day22_q_summary["file_name"] == dcac_file].iloc[0]
    q_final_dcac = float(q_summary_dcac["Q_final_Ah"])

    t_vmax_dcac_s = float(dcac_event["t_Vmax_detected_s"])
    q_vmax_dcac_ah = interp_q_at_time_day22(dcac_traj, t_vmax_dcac_s)

    t_acoff_dcac_s = dcac_event["t_AC_off_detected_s"]

    if is_finite_number(t_acoff_dcac_s):
        t_segmentB_start_s = min(float(t_vmax_dcac_s), float(t_acoff_dcac_s))
    else:
        t_segmentB_start_s = float(t_vmax_dcac_s)

    q_segmentB_start_ah = interp_q_at_time_day22(dcac_traj, t_segmentB_start_s)

    q_ordering_status = q_boundary_ordering_status_day22(
        q_segmentB_start_ah=q_segmentB_start_ah,
        q_vmax_dc_ah=q_vmax_dc_ah,
    )
    segment_framework_status = segment_framework_status_from_ordering_day22(q_ordering_status)

    if is_finite_number(dcac_event["AC_off_lag_s"]) and float(dcac_event["AC_off_lag_s"]) < 0:
        segment_framework_status = SEGMENT_FRAMEWORK_AC_OFF_PRECEDES_VMAX

    final_status = final_q_status_day22(q_final_dc, q_final_dcac)

    common = compute_common_anchors_day22(
        q_final_dc_ah=q_final_dc,
        q_final_dcac_ah=q_final_dcac,
        q_nom_ah=Q_NOM_AH,
    )

    q80_nominal_ah = Q80_NOMINAL_AH
    q90_nominal_ah = Q90_NOMINAL_AH
    q80_common_ah = common["Q80_common_Ah"]
    q90_common_ah = common["Q90_common_Ah"]

    q80_nominal_segment = assign_segment_by_q_day22(
        q80_nominal_ah,
        q_segmentB_start_ah,
        q_vmax_dc_ah,
        q_final_dc,
        q_final_dcac,
    )
    q90_nominal_segment = assign_segment_by_q_day22(
        q90_nominal_ah,
        q_segmentB_start_ah,
        q_vmax_dc_ah,
        q_final_dc,
        q_final_dcac,
    )
    q80_common_segment = assign_segment_by_q_day22(
        q80_common_ah,
        q_segmentB_start_ah,
        q_vmax_dc_ah,
        q_final_dc,
        q_final_dcac,
    )
    q90_common_segment = assign_segment_by_q_day22(
        q90_common_ah,
        q_segmentB_start_ah,
        q_vmax_dc_ah,
        q_final_dc,
        q_final_dcac,
    )

    row = {
        "protocol_pair": f"{dc_inv_day22['protocol_label']} vs {inv_row['protocol_label']}",
        "protocol_label_DC": dc_inv_day22["protocol_label"],
        "protocol_label_DCAC": inv_row["protocol_label"],
        "file_name_DC": dc_file_day22,
        "file_name_DCAC": dcac_file,

        "Q_nom_Ah": Q_NOM_AH,

        "Q_final_DC_Ah": q_final_dc,
        "Q_final_DCAC_Ah": q_final_dcac,
        "Q_final_diff_Ah": q_final_dc - q_final_dcac,
        "Q_final_diff_status": final_status,

        "t_Vmax_DC_s": t_vmax_dc_s,
        "t_Vmax_DCAC_s": t_vmax_dcac_s,
        "t_AC_off_DCAC_s": float(t_acoff_dcac_s) if is_finite_number(t_acoff_dcac_s) else np.nan,
        "AC_off_lag_s": float(dcac_event["AC_off_lag_s"]) if is_finite_number(dcac_event["AC_off_lag_s"]) else np.nan,
        "t_segmentB_start_s": t_segmentB_start_s,

        "Q_Vmax_DC_Ah": q_vmax_dc_ah,
        "Q_Vmax_DCAC_Ah": q_vmax_dcac_ah,
        "Q_segmentB_start_Ah": q_segmentB_start_ah,
        "segment_A_Q_hi_definition": "Q_segmentB_start_Ah",

        "Q_Vmax_shift_Ah": q_vmax_dc_ah - q_vmax_dcac_ah,
        "Q_Vmax_ordering_status": q_ordering_status,
        "segment_framework_status": segment_framework_status,

        "Q80_nominal_Ah": q80_nominal_ah,
        "Q90_nominal_Ah": q90_nominal_ah,
        "Q80_common_Ah": q80_common_ah,
        "Q90_common_Ah": q90_common_ah,

        "Q80_nominal_fraction_of_Q_nom": Q80_NOMINAL_FRACTION_OF_Q_NOM,
        "Q90_nominal_fraction_of_Q_nom": Q90_NOMINAL_FRACTION_OF_Q_NOM,
        "Q80_common_fraction_of_Q_nom": common["Q80_common_fraction_of_Q_nom"],
        "Q90_common_fraction_of_Q_nom": common["Q90_common_fraction_of_Q_nom"],

        "Q80_nominal_segment": q80_nominal_segment,
        "Q90_nominal_segment": q90_nominal_segment,
        "Q80_common_segment": q80_common_segment,
        "Q90_common_segment": q90_common_segment,

        "segment_A_Q_lo_Ah": SEGMENT_A_Q_LO_AH,
        "segment_A_Q_hi_Ah": q_segmentB_start_ah,
        "segment_A_Q_grid_count": segment_A_grid_count_day22(
            segment_A_Q_lo_Ah=SEGMENT_A_Q_LO_AH,
            segment_A_Q_hi_Ah=q_segmentB_start_ah,
        ),
    }

    segment_rows.append(row)

df_day22_segment_assignment = pd.DataFrame(segment_rows)
OUT_DAY22A_SEGMENT_ASSIGNMENT = DATA_DIR / "day22A_step4_MJ1_0p3C_0p4C_segment_assignment.csv"
df_day22_segment_assignment.to_csv(OUT_DAY22A_SEGMENT_ASSIGNMENT, index=False)

print(f"[OK] Wrote Day22A segment assignment audit: {OUT_DAY22A_SEGMENT_ASSIGNMENT}")

display_cols = [
    "protocol_pair",
    "Q_final_diff_status",
    "t_Vmax_DC_s",
    "t_Vmax_DCAC_s",
    "Q_Vmax_DC_Ah",
    "Q_Vmax_DCAC_Ah",
    "Q_Vmax_shift_Ah",
    "Q_segmentB_start_Ah",
    "Q_Vmax_ordering_status",
    "segment_framework_status",
    "Q80_nominal_Ah",
    "Q80_nominal_segment",
    "Q90_nominal_Ah",
    "Q90_nominal_segment",
    "Q80_common_Ah",
    "Q80_common_segment",
    "Q90_common_Ah",
    "Q90_common_segment",
    "segment_A_Q_grid_count",
]

print(df_day22_segment_assignment[display_cols].to_string(index=False))


# =============================================================================
# 6.4 Hard guards
# =============================================================================

bad_framework = df_day22_segment_assignment[
    df_day22_segment_assignment["segment_framework_status"].isin([
        SEGMENT_FRAMEWORK_ORDERING_VIOLATED,
        SEGMENT_FRAMEWORK_AC_OFF_PRECEDES_VMAX,
        SEGMENT_FRAMEWORK_UNRESOLVED,
    ])
]

if len(bad_framework) > 0:
    print("[warning] Day22A segment framework has non-OK rows:")
    print(bad_framework[display_cols].to_string(index=False))
    raise ValueError("Day22A segment framework status is not valid for all pairs.")

if (df_day22_segment_assignment["segment_A_Q_grid_count"] < Q_GRID_MIN_COUNT_SEGMENT_A).any():
    print("[warning] At least one Day22A pair has low Segment-A grid count.")
    print("          This does not stop Cell 6, but p95 residual guard may be unavailable later.")

print("[OK] Cell 6 Day22A event-charge extraction and anchor segment assignment completed.")
print("[OK] No Δt_raw, Δt_geom, Δt_resid, or verdict performed.")

[OK] Loaded Day22A event audit: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step2_MJ1_0p3C_0p4C_event_acoff_audit.csv
[OK] Loaded Day22A final-Q pair audit: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step3_MJ1_0p3C_0p4C_finalQ_pair_audit.csv
[OK] Day22A DC reference file = MJ1_0p3C_DC_NGU201_raw.csv
[OK] t_Vmax_DC_s = 10458.000
[OK] Q_Vmax_DC_Ah = 2.962785
[OK] Wrote Day22A segment assignment audit: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step4_MJ1_0p3C_0p4C_segment_assignment.csv
              protocol_pair Q_final_diff_status  t_Vmax_DC_s  t_Vmax_DCAC_s  Q_Vmax_DC_Ah  Q_Vmax_DCAC_Ah  Q_Vmax_shift_Ah  Q_segmentB_start_Ah                 Q_Vmax_ordering_status segment_framework_status  Q80_nominal_Ah                    Q80_nominal_segment  Q90_nominal_Ah Q90_nominal_segment  Q80_common_Ah          Q80_common_segment  Q90_common_Ah                     Q90_common_segment  segment_A_Q_grid_count
0.3C DC vs 0.3C+0.4C 0.1tau  final_Q_consistent      10458.0     

In [14]:
# Day22A Cell 7 — Δt(Q), Segment-A residual, and fitted-waveform diagnostics
#
# Purpose:
# - Compute first-passage raw Δt(Q)
# - Compute prescribed-geometry Δt_geom(Q)
# - Compute prescribed Segment-A Δt_resid(Q)
# - Compute fitted-waveform diagnostic residual
# - Compare residual scale against Day22A self-consistency audit-resolution floor
#
# Explicitly NOT done here:
# - No final mechanism verdict
# - No threshold redefinition
# - No closure note

from scipy.optimize import least_squares

OUT_DAY22A_DTQ_LONG = DATA_DIR / "day22A_step5_MJ1_0p3C_0p4C_dtQ_segment_audit_long.csv"
OUT_DAY22A_DTQ_SUMMARY = DATA_DIR / "day22A_step5_MJ1_0p3C_0p4C_dtQ_segment_summary.csv"
OUT_DAY22A_FIT_DIAG_LONG = DATA_DIR / "day22A_step5A_MJ1_0p3C_0p4C_fitted_waveform_diagnostic_long.csv"
OUT_DAY22A_FIT_DIAG_SUMMARY = DATA_DIR / "day22A_step5A_MJ1_0p3C_0p4C_fitted_waveform_diagnostic_summary.csv"

if not OUT_DAY22A_SEGMENT_ASSIGNMENT.exists():
    raise FileNotFoundError(
        f"Day22A segment assignment missing: {OUT_DAY22A_SEGMENT_ASSIGNMENT}\n"
        "Run Day22A Cell 6 first."
    )

if not OUT_DAY22A_RESOLUTION_SUMMARY.exists():
    raise FileNotFoundError(
        f"Day22A resolution summary missing: {OUT_DAY22A_RESOLUTION_SUMMARY}\n"
        "Run Day22A Cell 5A first."
    )

df_day22_segment_assignment = pd.read_csv(OUT_DAY22A_SEGMENT_ASSIGNMENT)
df_day22_resolution_summary = pd.read_csv(OUT_DAY22A_RESOLUTION_SUMMARY)

DAY22A_RESOLUTION_P95_S = float(
    df_day22_resolution_summary["day22A_self_consistency_resolution_p95_s"].iloc[0]
)
DAY22A_RESOLUTION_MAX_S = float(
    df_day22_resolution_summary["day22A_self_consistency_resolution_max_s"].iloc[0]
)

print(f"[OK] Loaded Day22A segment assignment: {OUT_DAY22A_SEGMENT_ASSIGNMENT}")
print(f"[OK] Day22A self-consistency resolution p95 = {DAY22A_RESOLUTION_P95_S:.6f} s")
print(f"[OK] Day22A self-consistency resolution max = {DAY22A_RESOLUTION_MAX_S:.6f} s")


# =============================================================================
# 7.1 Geometry and fitting helpers
# =============================================================================

def prescribed_current_day22(
    t_s: np.ndarray,
    I_DC_A: float,
    I_AC_A: float,
    frequency_Hz: float,
    phase_rad: float = 0.0,
) -> np.ndarray:
    """Charge-positive prescribed current."""
    t = np.asarray(t_s, dtype=float)

    if float(I_AC_A) == 0.0 or float(frequency_Hz) == 0.0:
        return np.full(len(t), float(I_DC_A), dtype=float)

    omega = 2.0 * np.pi * float(frequency_Hz)
    return float(I_DC_A) + float(I_AC_A) * np.sin(omega * t + float(phase_rad))


def prescribed_geometry_Q_Ah_day22(
    t_s: np.ndarray,
    I_DC_A: float,
    I_AC_A: float,
    frequency_Hz: float,
    phase_rad: float = 0.0,
) -> np.ndarray:
    """
    Compute prescribed geometry Q(t) using strict-net signed integration.
    """
    i_geom = prescribed_current_day22(
        t_s=t_s,
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        frequency_Hz=frequency_Hz,
        phase_rad=phase_rad,
    )
    return integrate_strict_net_Q_Ah_day22(t_s, i_geom)


def estimate_geometry_phase_day22(
    df_traj: pd.DataFrame,
    I_DC_A: float,
    I_AC_A: float,
    frequency_Hz: float,
    t_acoff_candidate_s: float,
) -> dict[str, object]:
    """
    Estimate phase by least-squares grid search over phi.

    Model:
    I_Q(t) = I_DC + I_AC sin(2π f t + phi)

    Only phi is estimated here; I_DC, I_AC, f are fixed by protocol metadata.
    """
    if float(I_AC_A) <= 0 or float(frequency_Hz) <= 0:
        return {
            "geometry_phase_offset_rad": 0.0,
            "geometry_phase_offset_s": 0.0,
            "geometry_phase_reference_status": GEOM_PHASE_VERIFIED,
            "geometry_phase_fit_rmse_A": 0.0,
            "geometry_phase_fit_window_s": np.nan,
        }

    T_AC_s = compute_t_ac_s(frequency_Hz)
    fit_window_s = min(5.0 * T_AC_s, float(t_acoff_candidate_s))

    if fit_window_s < T_AC_s:
        return {
            "geometry_phase_offset_rad": np.nan,
            "geometry_phase_offset_s": np.nan,
            "geometry_phase_reference_status": GEOM_PHASE_UNRESOLVED,
            "geometry_phase_fit_rmse_A": np.nan,
            "geometry_phase_fit_window_s": fit_window_s,
        }

    t = df_traj["t_s"].to_numpy(dtype=float)
    i = df_traj["I_Q_A"].to_numpy(dtype=float)

    mask = (
        np.isfinite(t)
        & np.isfinite(i)
        & (t >= 0)
        & (t <= fit_window_s)
    )

    if mask.sum() < 10:
        return {
            "geometry_phase_offset_rad": np.nan,
            "geometry_phase_offset_s": np.nan,
            "geometry_phase_reference_status": GEOM_PHASE_UNRESOLVED,
            "geometry_phase_fit_rmse_A": np.nan,
            "geometry_phase_fit_window_s": fit_window_s,
        }

    tt = t[mask]
    ii = i[mask]

    phi_grid = np.linspace(-np.pi, np.pi, 2001)
    omega = 2.0 * np.pi * float(frequency_Hz)

    best_phi = np.nan
    best_rmse = np.inf

    for phi in phi_grid:
        pred = float(I_DC_A) + float(I_AC_A) * np.sin(omega * tt + phi)
        rmse = float(np.sqrt(np.mean((ii - pred) ** 2)))
        if rmse < best_rmse:
            best_rmse = rmse
            best_phi = float(phi)

    phase_offset_s = best_phi / omega if omega > 0 else np.nan

    status = GEOM_PHASE_VERIFIED if abs(best_phi) <= 0.05 else GEOM_PHASE_ESTIMATED

    return {
        "geometry_phase_offset_rad": best_phi,
        "geometry_phase_offset_s": phase_offset_s,
        "geometry_phase_reference_status": status,
        "geometry_phase_fit_rmse_A": best_rmse,
        "geometry_phase_fit_window_s": fit_window_s,
    }


def fit_sine_current_segment_A_day22(
    df_traj: pd.DataFrame,
    t_hi_s: float,
    I0_init_A: float,
    A_init_A: float,
    f_init_Hz: float,
    phi_init_rad: float = 0.0,
) -> dict[str, object]:
    """
    Diagnostic fitted waveform:
    I(t) = I0 + A sin(2π f t + phi)

    This is diagnostic only.
    It does not replace the formal prescribed-geometry residual.
    """
    t = df_traj["t_s"].to_numpy(dtype=float)
    i = df_traj["I_Q_A"].to_numpy(dtype=float)

    mask = (
        np.isfinite(t)
        & np.isfinite(i)
        & (t >= 0)
        & (t <= float(t_hi_s))
    )

    if mask.sum() < 20:
        return {
            "fit_status": "unresolved_insufficient_samples",
            "I0_fit_A": np.nan,
            "A_fit_A": np.nan,
            "f_fit_Hz": np.nan,
            "phi_fit_rad": np.nan,
            "fit_rmse_A": np.nan,
            "fit_n": int(mask.sum()),
        }

    tt = t[mask]
    ii = i[mask]
    tt_rel = tt - tt[0]

    if not np.isfinite(f_init_Hz) or f_init_Hz <= 0:
        return {
            "fit_status": "unresolved_invalid_initial_frequency",
            "I0_fit_A": np.nan,
            "A_fit_A": np.nan,
            "f_fit_Hz": np.nan,
            "phi_fit_rad": np.nan,
            "fit_rmse_A": np.nan,
            "fit_n": int(mask.sum()),
        }

    def model(params):
        I0, A, f, phi = params
        return I0 + A * np.sin(2.0 * np.pi * f * tt_rel + phi)

    def residual(params):
        return model(params) - ii

    x0 = np.array([
        float(I0_init_A),
        max(float(A_init_A), 1e-6),
        float(f_init_Hz),
        float(phi_init_rad),
    ])

    lower = np.array([
        -ONE_C_A,
        0.0,
        0.5 * float(f_init_Hz),
        -np.pi,
    ])

    upper = np.array([
        2.0 * ONE_C_A,
        2.0 * ONE_C_A,
        1.5 * float(f_init_Hz),
        np.pi,
    ])

    try:
        res = least_squares(
            residual,
            x0=x0,
            bounds=(lower, upper),
            max_nfev=5000,
            xtol=1e-12,
            ftol=1e-12,
            gtol=1e-12,
        )
    except Exception as exc:
        return {
            "fit_status": f"unresolved_fit_error:{type(exc).__name__}",
            "I0_fit_A": np.nan,
            "A_fit_A": np.nan,
            "f_fit_Hz": np.nan,
            "phi_fit_rad": np.nan,
            "fit_rmse_A": np.nan,
            "fit_n": int(mask.sum()),
        }

    I0_fit, A_fit, f_fit, phi_fit = res.x
    pred = model(res.x)
    rmse = float(np.sqrt(np.mean((pred - ii) ** 2)))

    return {
        "fit_status": "ok" if res.success else "fit_not_successful",
        "I0_fit_A": float(I0_fit),
        "A_fit_A": float(A_fit),
        "f_fit_Hz": float(f_fit),
        "phi_fit_rad": float(phi_fit),
        "fit_rmse_A": rmse,
        "fit_n": int(mask.sum()),
    }


def fitted_geometry_Q_Ah_day22(
    t_s: np.ndarray,
    I0_A: float,
    A_A: float,
    f_Hz: float,
    phi_rad: float,
) -> np.ndarray:
    """
    Diagnostic fitted-waveform Q(t).
    """
    t = np.asarray(t_s, dtype=float)
    if len(t) == 0:
        return np.array([], dtype=float)

    t_rel = t - t[0]
    i_fit = I0_A + A_A * np.sin(2.0 * np.pi * f_Hz * t_rel + phi_rad)
    return integrate_strict_net_Q_Ah_day22(t, i_fit)


def safe_mean_day22(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.mean(arr)) if len(arr) else np.nan


def safe_median_day22(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.median(arr)) if len(arr) else np.nan


def safe_max_day22(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.max(arr)) if len(arr) else np.nan


def safe_min_day22(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.min(arr)) if len(arr) else np.nan


def safe_max_abs_day22(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.max(np.abs(arr))) if len(arr) else np.nan


def safe_p95_abs_day22(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.percentile(np.abs(arr), 95)) if len(arr) else np.nan


def segment_A_above_floor_status_day22(max_abs_s, p95_abs_s, q_grid_count):
    if not is_finite_number(max_abs_s):
        return ABOVE_FLOOR_UNRESOLVED

    max_abs = abs(float(max_abs_s))

    if max_abs <= SEG_A_FLOOR_COMPATIBLE_THRESHOLD_S:
        return ABOVE_FLOOR_NO

    if max_abs < SEG_A_REOPEN_THRESHOLD_S:
        return ABOVE_FLOOR_INTERMEDIATE

    if q_grid_count < Q_GRID_MIN_COUNT_SEGMENT_A or not is_finite_number(p95_abs_s):
        return "above_floor_no_p95_guard"

    if abs(float(p95_abs_s)) < SEG_A_REOPEN_THRESHOLD_S:
        return ABOVE_FLOOR_SPIKE

    return ABOVE_FLOOR_YES


# =============================================================================
# 7.2 Build Δt(Q) audit
# =============================================================================

dtq_rows = []
fit_rows = []
summary_rows = []

dc_file = get_day22_dc_reference_file_name()
dc_traj = TRAJ22[dc_file]

t_dc = dc_traj["t_s"].to_numpy(dtype=float)
q_dc = dc_traj["Q_net_Ah"].to_numpy(dtype=float)

for _, seg_row in df_day22_segment_assignment.iterrows():
    pair = seg_row["protocol_pair"]
    dcac_file = seg_row["file_name_DCAC"]
    dcac_traj = TRAJ22[dcac_file]
    inv_dcac = get_day22_inventory_row(dcac_file)

    t_dcac = dcac_traj["t_s"].to_numpy(dtype=float)
    q_dcac = dcac_traj["Q_net_Ah"].to_numpy(dtype=float)

    I_DC_A = float(inv_dcac["DC_C"]) * ONE_C_A
    I_AC_A = float(inv_dcac["AC_C"]) * ONE_C_A
    f_hz = float(inv_dcac["frequency_Hz"])

    phase_result = estimate_geometry_phase_day22(
        df_traj=dcac_traj,
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        frequency_Hz=f_hz,
        t_acoff_candidate_s=float(seg_row["t_segmentB_start_s"]),
    )

    phase_rad = phase_result["geometry_phase_offset_rad"]
    if not is_finite_number(phase_rad):
        phase_rad = 0.0

    # Prescribed geometry
    q_geom_dc = prescribed_geometry_Q_Ah_day22(
        t_s=t_dc,
        I_DC_A=I_DC_A,
        I_AC_A=0.0,
        frequency_Hz=0.0,
        phase_rad=0.0,
    )

    q_geom_dcac = prescribed_geometry_Q_Ah_day22(
        t_s=t_dcac,
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        frequency_Hz=f_hz,
        phase_rad=phase_rad,
    )

    # Fitted diagnostic geometry
    fit = fit_sine_current_segment_A_day22(
        df_traj=dcac_traj,
        t_hi_s=float(seg_row["t_segmentB_start_s"]),
        I0_init_A=I_DC_A,
        A_init_A=I_AC_A,
        f_init_Hz=f_hz,
        phi_init_rad=phase_rad,
    )

    if fit["fit_status"] == "ok":
        q_geom_dcac_fit = fitted_geometry_Q_Ah_day22(
            t_s=t_dcac,
            I0_A=fit["I0_fit_A"],
            A_A=fit["A_fit_A"],
            f_Hz=fit["f_fit_Hz"],
            phi_rad=fit["phi_fit_rad"],
        )
    else:
        q_geom_dcac_fit = np.full(len(t_dcac), np.nan)

    q_seg_A_lo = SEGMENT_A_Q_LO_AH
    q_seg_A_hi = float(seg_row["segment_A_Q_hi_Ah"])
    q_seg_B_lo = q_seg_A_hi
    q_seg_B_hi = float(seg_row["Q_Vmax_DC_Ah"])
    q_seg_D_lo = q_seg_B_hi
    q_seg_D_hi = min(float(seg_row["Q_final_DC_Ah"]), float(seg_row["Q_final_DCAC_Ah"]))

    grid_A = make_fixed_Q_grid_day22(q_seg_A_lo, q_seg_A_hi, Q_GRID_STEP_AH)
    grid_B = make_fixed_Q_grid_day22(q_seg_B_lo + Q_GRID_STEP_AH, q_seg_B_hi, Q_GRID_STEP_AH)
    grid_D = make_fixed_Q_grid_day22(q_seg_D_lo + Q_GRID_STEP_AH, q_seg_D_hi, Q_GRID_STEP_AH)

    anchor_targets = {
        "Q80_nominal": float(seg_row["Q80_nominal_Ah"]),
        "Q90_nominal": float(seg_row["Q90_nominal_Ah"]),
        "Q80_common": float(seg_row["Q80_common_Ah"]),
        "Q90_common": float(seg_row["Q90_common_Ah"]),
    }

    anchor_segment_map = {
        "Q80_nominal": seg_row["Q80_nominal_segment"],
        "Q90_nominal": seg_row["Q90_nominal_segment"],
        "Q80_common": seg_row["Q80_common_segment"],
        "Q90_common": seg_row["Q90_common_segment"],
    }

    def add_row(q_target, q_label, segment_label, is_anchor):
        t_dc_q = first_passage_time_from_Q_day22(q_target, t_dc, q_dc)
        t_dcac_q = first_passage_time_from_Q_day22(q_target, t_dcac, q_dcac)
        dt_raw = (
            t_dc_q - t_dcac_q
            if is_finite_number(t_dc_q) and is_finite_number(t_dcac_q)
            else np.nan
        )

        if segment_label == SEGMENT_A:
            t_geom_dc_q = first_passage_time_from_Q_day22(q_target, t_dc, q_geom_dc)
            t_geom_dcac_q = first_passage_time_from_Q_day22(q_target, t_dcac, q_geom_dcac)

            dt_geom = (
                t_geom_dc_q - t_geom_dcac_q
                if is_finite_number(t_geom_dc_q) and is_finite_number(t_geom_dcac_q)
                else np.nan
            )

            dt_resid = (
                dt_raw - dt_geom
                if is_finite_number(dt_raw) and is_finite_number(dt_geom)
                else np.nan
            )

            t_geom_dcac_fit_q = first_passage_time_from_Q_day22(
                q_target,
                t_dcac,
                q_geom_dcac_fit,
            )
            dt_geom_fit = (
                t_geom_dc_q - t_geom_dcac_fit_q
                if is_finite_number(t_geom_dc_q) and is_finite_number(t_geom_dcac_fit_q)
                else np.nan
            )

            dt_resid_fit = (
                dt_raw - dt_geom_fit
                if is_finite_number(dt_raw) and is_finite_number(dt_geom_fit)
                else np.nan
            )
        else:
            t_geom_dc_q = np.nan
            t_geom_dcac_q = np.nan
            dt_geom = np.nan
            dt_resid = np.nan
            dt_geom_fit = np.nan
            dt_resid_fit = np.nan

        dtq_rows.append({
            "protocol_pair": pair,
            "protocol_label_DCAC": seg_row["protocol_label_DCAC"],
            "file_name_DCAC": dcac_file,
            "Q_label": q_label,
            "Q_Ah": float(q_target),
            "is_anchor": bool(is_anchor),
            "segment_label": segment_label,
            "t_DC_s": t_dc_q,
            "t_DCAC_s": t_dcac_q,
            "dt_raw_s": dt_raw,
            "t_geom_DC_s": t_geom_dc_q,
            "t_geom_DCAC_s": t_geom_dcac_q,
            "dt_geom_s": dt_geom,
            "dt_resid_s": dt_resid,
            "dt_geom_fit_s": dt_geom_fit,
            "dt_resid_fit_s": dt_resid_fit,
            "geometry_phase_offset_rad": phase_result["geometry_phase_offset_rad"],
            "geometry_phase_offset_s": phase_result["geometry_phase_offset_s"],
            "geometry_phase_reference_status": phase_result["geometry_phase_reference_status"],
            "geometry_phase_fit_rmse_A": phase_result["geometry_phase_fit_rmse_A"],
            "geometry_phase_fit_window_s": phase_result["geometry_phase_fit_window_s"],
            "fit_status": fit["fit_status"],
            "I0_fit_A": fit["I0_fit_A"],
            "A_fit_A": fit["A_fit_A"],
            "f_meta_Hz": f_hz,
            "f_fit_Hz": fit["f_fit_Hz"],
            "f_fit_rel_error_ppm": (
                (fit["f_fit_Hz"] - f_hz) / f_hz * 1e6
                if fit["fit_status"] == "ok" and f_hz > 0
                else np.nan
            ),
            "fit_rmse_A": fit["fit_rmse_A"],
        })

    for q in grid_A:
        add_row(q, "SegmentA_grid", SEGMENT_A, False)

    for q in grid_B:
        add_row(q, "SegmentB_grid", SEGMENT_B, False)

    for q in grid_D:
        add_row(q, "SegmentD_grid", SEGMENT_D, False)

    for label, q in anchor_targets.items():
        add_row(q, label, anchor_segment_map[label], True)

    df_pair = pd.DataFrame([r for r in dtq_rows if r["protocol_pair"] == pair])

    segA = df_pair[(df_pair["segment_label"] == SEGMENT_A) & (~df_pair["is_anchor"])]
    segB = df_pair[(df_pair["segment_label"] == SEGMENT_B) & (~df_pair["is_anchor"])]
    segD = df_pair[(df_pair["segment_label"] == SEGMENT_D) & (~df_pair["is_anchor"])]

    segA_resid = segA["dt_resid_s"].dropna().to_numpy(dtype=float)
    segA_resid_fit = segA["dt_resid_fit_s"].dropna().to_numpy(dtype=float)
    segA_raw = segA["dt_raw_s"].dropna().to_numpy(dtype=float)
    segB_raw = segB["dt_raw_s"].dropna().to_numpy(dtype=float)
    segD_raw = segD["dt_raw_s"].dropna().to_numpy(dtype=float)

    segA_grid_count = int(len(segA))

    segA_resid_mean = safe_mean_day22(segA_resid)
    segA_resid_max_abs = safe_max_abs_day22(segA_resid)
    segA_resid_p95_abs = safe_p95_abs_day22(segA_resid) if segA_grid_count >= Q_GRID_MIN_COUNT_SEGMENT_A else np.nan

    segA_fit_p95_abs = safe_p95_abs_day22(segA_resid_fit) if segA_grid_count >= Q_GRID_MIN_COUNT_SEGMENT_A else np.nan
    segA_fit_max_abs = safe_max_abs_day22(segA_resid_fit)

    segA_status = segment_A_above_floor_status_day22(
        max_abs_s=segA_resid_max_abs,
        p95_abs_s=segA_resid_p95_abs,
        q_grid_count=segA_grid_count,
    )

    audit_resolution_status = (
        "prescribed_p95_above_self_consistency_resolution"
        if is_finite_number(segA_resid_p95_abs) and segA_resid_p95_abs > DAY22A_RESOLUTION_P95_S
        else "prescribed_p95_below_or_equal_self_consistency_resolution"
    )

    fitted_resolution_status = (
        "fitted_p95_above_self_consistency_resolution"
        if is_finite_number(segA_fit_p95_abs) and segA_fit_p95_abs > DAY22A_RESOLUTION_P95_S
        else "fitted_p95_below_or_equal_self_consistency_resolution"
    )

    anchor_rows = df_pair[df_pair["is_anchor"]].copy()
    anchor_dt = {
        row["Q_label"]: row["dt_raw_s"]
        for _, row in anchor_rows.iterrows()
    }

    segmentD_anchor_dt = anchor_rows.loc[
        anchor_rows["segment_label"] == SEGMENT_D,
        "dt_raw_s",
    ].dropna().to_list()

    late_cv_status = (
        LATE_CV_NOT_REQUIRED
        if len(segmentD_anchor_dt) == 0
        else (
            LATE_CV_SATISFIED
            if all(x > LATE_CV_PRESERVATION_THRESHOLD_S for x in segmentD_anchor_dt)
            and safe_median_day22(segD_raw) > LATE_CV_PRESERVATION_THRESHOLD_S
            else LATE_CV_NOT_SATISFIED
        )
    )

    summary_rows.append({
        "protocol_pair": pair,
        "protocol_label_DCAC": seg_row["protocol_label_DCAC"],

        "geometry_phase_offset_rad": phase_result["geometry_phase_offset_rad"],
        "geometry_phase_offset_s": phase_result["geometry_phase_offset_s"],
        "geometry_phase_reference_status": phase_result["geometry_phase_reference_status"],
        "geometry_phase_fit_rmse_A": phase_result["geometry_phase_fit_rmse_A"],
        "geometry_phase_fit_window_s": phase_result["geometry_phase_fit_window_s"],

        "fit_status": fit["fit_status"],
        "I0_fit_A": fit["I0_fit_A"],
        "A_fit_A": fit["A_fit_A"],
        "f_meta_Hz": f_hz,
        "f_fit_Hz": fit["f_fit_Hz"],
        "f_fit_rel_error_ppm": (
            (fit["f_fit_Hz"] - f_hz) / f_hz * 1e6
            if fit["fit_status"] == "ok" and f_hz > 0
            else np.nan
        ),
        "fit_rmse_A": fit["fit_rmse_A"],
        "fit_rmse_fraction_of_A_fit": (
            fit["fit_rmse_A"] / abs(fit["A_fit_A"])
            if fit["fit_status"] == "ok" and is_finite_number(fit["A_fit_A"]) and abs(fit["A_fit_A"]) > 0
            else np.nan
        ),

        "dt_Q80_nominal_raw_s": anchor_dt.get("Q80_nominal", np.nan),
        "dt_Q90_nominal_raw_s": anchor_dt.get("Q90_nominal", np.nan),
        "dt_Q80_common_raw_s": anchor_dt.get("Q80_common", np.nan),
        "dt_Q90_common_raw_s": anchor_dt.get("Q90_common", np.nan),

        "segment_A_Q_lo_Ah": q_seg_A_lo,
        "segment_A_Q_hi_Ah": q_seg_A_hi,
        "segment_A_Q_grid_count": segA_grid_count,
        "segment_A_dt_raw_median_s": safe_median_day22(segA_raw),
        "segment_A_dt_raw_max_s": safe_max_day22(segA_raw),
        "segment_A_dt_resid_mean_s": segA_resid_mean,
        "segment_A_dt_resid_max_abs_s": segA_resid_max_abs,
        "segment_A_dt_resid_p95_abs_s": segA_resid_p95_abs,
        "segment_A_resid_floor_s": MJ1_FLOOR_MAX_ABS_S,
        "segment_A_resid_floor_type": MJ1_FLOOR_TYPE,
        "segment_A_resid_floor_n": MJ1_FLOOR_N,
        "segment_A_above_floor_status": segA_status,

        "segment_A_dt_resid_fit_p95_abs_s": segA_fit_p95_abs,
        "segment_A_dt_resid_fit_max_abs_s": segA_fit_max_abs,

        "day22A_self_consistency_resolution_p95_s": DAY22A_RESOLUTION_P95_S,
        "day22A_self_consistency_resolution_max_s": DAY22A_RESOLUTION_MAX_S,
        "audit_resolution_status": audit_resolution_status,
        "fitted_resolution_status": fitted_resolution_status,

        "segment_B_Q_lo_Ah": q_seg_B_lo,
        "segment_B_Q_hi_Ah": q_seg_B_hi,
        "segment_B_dt_raw_median_s": safe_median_day22(segB_raw),
        "segment_B_dt_raw_max_s": safe_max_day22(segB_raw),

        "segment_D_Q_lo_Ah": q_seg_D_lo,
        "segment_D_Q_hi_Ah": q_seg_D_hi,
        "segment_D_dt_raw_min_s": safe_min_day22(segD_raw),
        "segment_D_dt_raw_median_s": safe_median_day22(segD_raw),
        "segment_D_dt_raw_max_s": safe_max_day22(segD_raw),
        "late_CV_preservation_threshold_s": LATE_CV_PRESERVATION_THRESHOLD_S,
        "late_CV_preservation_satisfied": late_cv_status,
    })


df_day22_dtq_long = pd.DataFrame(dtq_rows)
df_day22_dtq_summary = pd.DataFrame(summary_rows)

OUT_DAY22A_DTQ_LONG = DATA_DIR / "day22A_step5_MJ1_0p3C_0p4C_dtQ_segment_audit_long.csv"
OUT_DAY22A_DTQ_SUMMARY = DATA_DIR / "day22A_step5_MJ1_0p3C_0p4C_dtQ_segment_summary.csv"

df_day22_dtq_long.to_csv(OUT_DAY22A_DTQ_LONG, index=False)
df_day22_dtq_summary.to_csv(OUT_DAY22A_DTQ_SUMMARY, index=False)

print(f"[OK] Wrote Day22A Δt(Q) long table: {OUT_DAY22A_DTQ_LONG}")
print(f"[OK] Wrote Day22A Δt(Q) summary: {OUT_DAY22A_DTQ_SUMMARY}")

display_cols = [
    "protocol_pair",
    "geometry_phase_reference_status",
    "geometry_phase_offset_rad",
    "fit_status",
    "fit_rmse_fraction_of_A_fit",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "segment_A_Q_grid_count",
    "segment_A_dt_resid_mean_s",
    "segment_A_dt_resid_max_abs_s",
    "segment_A_dt_resid_p95_abs_s",
    "segment_A_above_floor_status",
    "segment_A_dt_resid_fit_p95_abs_s",
    "audit_resolution_status",
    "fitted_resolution_status",
    "segment_B_dt_raw_median_s",
    "segment_D_dt_raw_median_s",
    "late_CV_preservation_satisfied",
]

print(df_day22_dtq_summary[display_cols].to_string(index=False))


# =============================================================================
# 7.3 Hard guards
# =============================================================================

if (df_day22_dtq_summary["geometry_phase_reference_status"] == GEOM_PHASE_UNRESOLVED).any():
    raise ValueError("Day22A geometry phase unresolved for at least one pair.")

if (df_day22_dtq_summary["segment_A_Q_grid_count"] < Q_GRID_MIN_COUNT_SEGMENT_A).any():
    print("[warning] Segment-A Q-grid count below p95 requirement for at least one pair.")

print("[OK] Cell 7 Day22A Δt(Q), residual, and diagnostic audit completed.")
print("[OK] No final verdict performed.")

[OK] Loaded Day22A segment assignment: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step4_MJ1_0p3C_0p4C_segment_assignment.csv
[OK] Day22A self-consistency resolution p95 = 1.414811 s
[OK] Day22A self-consistency resolution max = 8.256878 s
[OK] Wrote Day22A Δt(Q) long table: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step5_MJ1_0p3C_0p4C_dtQ_segment_audit_long.csv
[OK] Wrote Day22A Δt(Q) summary: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step5_MJ1_0p3C_0p4C_dtQ_segment_summary.csv
              protocol_pair geometry_phase_reference_status  geometry_phase_offset_rad fit_status  fit_rmse_fraction_of_A_fit  dt_Q80_common_raw_s  dt_Q90_common_raw_s  segment_A_Q_grid_count  segment_A_dt_resid_mean_s  segment_A_dt_resid_max_abs_s  segment_A_dt_resid_p95_abs_s                    segment_A_above_floor_status  segment_A_dt_resid_fit_p95_abs_s                          audit_resolution_status                              fitted_resolution_status  segment_B_dt_raw_median

In [15]:
# Day22A Cell 8 — Formal verdict with low-amplitude resolution caveats
#
# Purpose:
# - Apply Day22A formal verdict logic to 0.3C+0.4C group
# - Preserve prescribed-geometry Segment-A residual status
# - Attach audit-resolution and fitted-waveform diagnostic caveats
#
# Explicitly NOT done here:
# - No threshold redefinition
# - No claim of strict effect disappearance
# - No fitted residual as formal replacement for prescribed residual

OUT_DAY22A_VERDICT = DATA_DIR / "day22A_step6_MJ1_0p3C_0p4C_mechanism_verdict.csv"

required_day22_verdict_files = [
    OUT_DAY22A_SEGMENT_ASSIGNMENT,
    OUT_DAY22A_DTQ_SUMMARY,
    OUT_DAY22A_FINALQ_PAIR_AUDIT,
    OUT_DAY22A_RESOLUTION_SUMMARY,
]

for p in required_day22_verdict_files:
    if not p.exists():
        raise FileNotFoundError(f"Required Day22A verdict input missing: {p}")

df_day22_segment_assignment = pd.read_csv(OUT_DAY22A_SEGMENT_ASSIGNMENT)
df_day22_dtq_summary = pd.read_csv(OUT_DAY22A_DTQ_SUMMARY)
df_day22_finalq_pairs = pd.read_csv(OUT_DAY22A_FINALQ_PAIR_AUDIT)
df_day22_resolution_summary = pd.read_csv(OUT_DAY22A_RESOLUTION_SUMMARY)


def append_caveat_day22(existing, new):
    if new is None or str(new).strip() == "":
        return "" if pd.isna(existing) else str(existing)

    if existing is None or pd.isna(existing) or str(existing).strip() in ["", "nan", "NaN", "<NA>"]:
        items = []
    else:
        items = [x.strip() for x in str(existing).split(";") if x.strip()]

    if new not in items:
        items.append(new)

    return ";".join(items)


def one_day22_row(df: pd.DataFrame, mask, label: str) -> pd.Series:
    rows = df.loc[mask]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one row for {label}, found {len(rows)}")
    return rows.iloc[0]


def day22_formal_verdict(seg_row: pd.Series, dtq_row: pd.Series) -> dict:
    """
    Formal verdict uses prescribed-geometry Segment-A residual only.
    Diagnostic fitted-waveform fields are added as caveats, not as formal replacement.
    """
    q80_common_segment = seg_row["Q80_common_segment"]
    q90_common_segment = seg_row["Q90_common_segment"]
    q80_nominal_segment = seg_row["Q80_nominal_segment"]
    q90_nominal_segment = seg_row["Q90_nominal_segment"]

    anchor_segments = [
        q80_common_segment,
        q90_common_segment,
        q80_nominal_segment,
        q90_nominal_segment,
    ]

    any_A_anchor = any(s == SEGMENT_A for s in anchor_segments)
    all_common_B = (
        q80_common_segment == SEGMENT_B
        and q90_common_segment == SEGMENT_B
    )

    segA_status = dtq_row["segment_A_above_floor_status"]

    if segA_status == ABOVE_FLOOR_NO:
        if all_common_B:
            return {
                "evidence_status": "partial_support",
                "mechanism_verdict": "interpretation_A_weak_boundary_control_state_supported",
                "interpretation_class": "boundary_control_state_mediated_gain_low_amplitude",
            }
        return {
            "evidence_status": "partial_support",
            "mechanism_verdict": "interpretation_A_weak_non_geometric_A_not_supported",
            "interpretation_class": "floor_compatible_mixed_anchor_distribution",
        }

    if segA_status == ABOVE_FLOOR_INTERMEDIATE:
        return {
            "evidence_status": "ambiguous",
            "mechanism_verdict": "ambiguous_intermediate_prescribed_segment_A_residual",
            "interpretation_class": "intermediate_prescribed_residual_near_audit_resolution",
        }

    if segA_status == ABOVE_FLOOR_SPIKE:
        return {
            "evidence_status": "ambiguous",
            "mechanism_verdict": "ambiguous_spike_or_transition_artifact",
            "interpretation_class": "spike_or_transition_artifact",
        }

    if segA_status == ABOVE_FLOOR_YES:
        if any_A_anchor:
            return {
                "evidence_status": "formal_reopened_with_diagnostic_caveat",
                "mechanism_verdict": "interpretation_B_formally_reopened_by_prescribed_segment_A_residual",
                "interpretation_class": "formal_prescribed_segment_A_above_floor_with_inwindow_A_anchor",
            }
        return {
            "evidence_status": "ambiguous",
            "mechanism_verdict": "ambiguous_above_floor_without_inwindow_A_anchor",
            "interpretation_class": "segmentA_above_floor_without_inwindow_A_anchor",
        }

    return {
        "evidence_status": "ambiguous",
        "mechanism_verdict": "ambiguous_unclassified_segment_A_status",
        "interpretation_class": "unclassified_segment_A_status",
    }


def diagnostic_caveats_day22(dtq_row: pd.Series) -> str:
    caveat = ""

    caveat = append_caveat_day22(caveat, DAY22A_RESOLUTION_STATUS_NO_REPEAT)
    caveat = append_caveat_day22(caveat, DAY22A_LOW_AMPLITUDE_VERDICT_LIMITATION)

    if str(dtq_row["audit_resolution_status"]) == "prescribed_p95_above_self_consistency_resolution":
        caveat = append_caveat_day22(caveat, "prescribed_p95_above_self_consistency_resolution")
    else:
        caveat = append_caveat_day22(caveat, "prescribed_p95_below_or_equal_self_consistency_resolution")

    fit_frac = dtq_row["fit_rmse_fraction_of_A_fit"]
    fit_usable = is_finite_number(fit_frac) and float(fit_frac) <= 0.10

    if not fit_usable:
        caveat = append_caveat_day22(caveat, "diagnostic_fit_unreliable")
    else:
        caveat = append_caveat_day22(caveat, "diagnostic_fit_usable")

    if str(dtq_row["fitted_resolution_status"]) == "fitted_p95_below_or_equal_self_consistency_resolution":
        caveat = append_caveat_day22(caveat, "diagnostic_fitted_p95_below_or_equal_audit_resolution")
    else:
        caveat = append_caveat_day22(caveat, "diagnostic_fitted_p95_above_audit_resolution")

    if (
        is_finite_number(dtq_row["segment_A_dt_resid_fit_p95_abs_s"])
        and is_finite_number(DAY22A_RESOLUTION_P95_S)
        and float(dtq_row["segment_A_dt_resid_fit_p95_abs_s"]) <= DAY22A_RESOLUTION_P95_S
        and fit_usable
    ):
        caveat = append_caveat_day22(caveat, "diagnostic_waveform_geometry_mismatch_likely")

    return caveat


verdict_rows = []

for _, seg_row in df_day22_segment_assignment.iterrows():
    pair = seg_row["protocol_pair"]

    dtq_row = one_day22_row(
        df_day22_dtq_summary,
        df_day22_dtq_summary["protocol_pair"] == pair,
        f"dtq_summary:{pair}",
    )

    finalq_row = one_day22_row(
        df_day22_finalq_pairs,
        df_day22_finalq_pairs["protocol_pair"] == pair,
        f"finalq:{pair}",
    )

    formal = day22_formal_verdict(seg_row, dtq_row)

    caveat = diagnostic_caveats_day22(dtq_row)

    if finalq_row["Q_final_diff_status"] == FINAL_Q_MISMATCH_WARNING:
        caveat = append_caveat_day22(caveat, "asymmetric_final_Q")

    if dtq_row["late_CV_preservation_satisfied"] == LATE_CV_SATISFIED:
        caveat = append_caveat_day22(caveat, "late_CV_preservation_satisfied")

    row = {
        "source_type": SOURCE_TYPE_MJ1,
        "cell_or_param_set": CELL_ID,
        "group_id": DAY22A_GROUP_ID,
        "protocol_pair": pair,
        "protocol_label_DC": seg_row["protocol_label_DC"],
        "protocol_label_DCAC": seg_row["protocol_label_DCAC"],

        "Q_final_diff_status": finalq_row["Q_final_diff_status"],
        "Q_final_diff_mAh": finalq_row["Q_final_diff_mAh"],

        "Q_Vmax_DC_Ah": seg_row["Q_Vmax_DC_Ah"],
        "Q_Vmax_DCAC_Ah": seg_row["Q_Vmax_DCAC_Ah"],
        "Q_Vmax_shift_Ah": seg_row["Q_Vmax_shift_Ah"],
        "segment_framework_status": seg_row["segment_framework_status"],

        "Q80_common_Ah": seg_row["Q80_common_Ah"],
        "Q90_common_Ah": seg_row["Q90_common_Ah"],
        "Q80_common_segment": seg_row["Q80_common_segment"],
        "Q90_common_segment": seg_row["Q90_common_segment"],
        "Q80_nominal_segment": seg_row["Q80_nominal_segment"],
        "Q90_nominal_segment": seg_row["Q90_nominal_segment"],

        "dt_Q80_common_raw_s": dtq_row["dt_Q80_common_raw_s"],
        "dt_Q90_common_raw_s": dtq_row["dt_Q90_common_raw_s"],
        "segment_A_above_floor_status": dtq_row["segment_A_above_floor_status"],
        "segment_A_dt_resid_mean_s": dtq_row["segment_A_dt_resid_mean_s"],
        "segment_A_dt_resid_max_abs_s": dtq_row["segment_A_dt_resid_max_abs_s"],
        "segment_A_dt_resid_p95_abs_s": dtq_row["segment_A_dt_resid_p95_abs_s"],

        "segment_A_dt_resid_fit_p95_abs_s": dtq_row["segment_A_dt_resid_fit_p95_abs_s"],
        "segment_A_dt_resid_fit_max_abs_s": dtq_row["segment_A_dt_resid_fit_max_abs_s"],
        "fit_status": dtq_row["fit_status"],
        "fit_rmse_fraction_of_A_fit": dtq_row["fit_rmse_fraction_of_A_fit"],

        "day22A_self_consistency_resolution_p95_s": DAY22A_RESOLUTION_P95_S,
        "day22A_self_consistency_resolution_max_s": DAY22A_RESOLUTION_MAX_S,
        "audit_resolution_status": dtq_row["audit_resolution_status"],
        "fitted_resolution_status": dtq_row["fitted_resolution_status"],

        "segment_B_dt_raw_median_s": dtq_row["segment_B_dt_raw_median_s"],
        "segment_D_dt_raw_median_s": dtq_row["segment_D_dt_raw_median_s"],
        "late_CV_preservation_satisfied": dtq_row["late_CV_preservation_satisfied"],

        "evidence_status": formal["evidence_status"],
        "mechanism_verdict": formal["mechanism_verdict"],
        "interpretation_class": formal["interpretation_class"],
        "caveat": caveat,
    }

    verdict_rows.append(row)

df_day22_verdict = pd.DataFrame(verdict_rows)
df_day22_verdict.to_csv(OUT_DAY22A_VERDICT, index=False)

print(f"[OK] Wrote Day22A verdict table: {OUT_DAY22A_VERDICT}")

display_cols = [
    "protocol_pair",
    "evidence_status",
    "mechanism_verdict",
    "interpretation_class",
    "Q80_common_segment",
    "Q90_common_segment",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "segment_A_above_floor_status",
    "segment_A_dt_resid_p95_abs_s",
    "segment_A_dt_resid_fit_p95_abs_s",
    "audit_resolution_status",
    "fitted_resolution_status",
    "caveat",
]

print(df_day22_verdict[display_cols].to_string(index=False))

print("[OK] Cell 8 Day22A verdict completed.")
print("[OK] Formal disappearance claim remains prohibited without repeat-based noise floor.")

[OK] Wrote Day22A verdict table: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step6_MJ1_0p3C_0p4C_mechanism_verdict.csv
              protocol_pair                        evidence_status                                                   mechanism_verdict                                           interpretation_class          Q80_common_segment                     Q90_common_segment  dt_Q80_common_raw_s  dt_Q90_common_raw_s                    segment_A_above_floor_status  segment_A_dt_resid_p95_abs_s  segment_A_dt_resid_fit_p95_abs_s                          audit_resolution_status                              fitted_resolution_status                                                                                                                                                                                                                                                                                                                                      caveat
0.3C DC vs 0.3C+0.4

In [16]:
# Day22A Cell 9A — closure summary CSV
#
# Purpose:
# - Close Day22A in machine-readable form
# - No recomputation
# - No new thresholds
# - No new verdict logic

OUT_DAY22A_CLOSURE_CSV = DATA_DIR / "day22A_step7_closure_summary.csv"
OUT_DAY22A_CLOSURE_MD = DATA_DIR / "day22A_step7_closure_note.md"

required_day22_closure_files = [
    OUT_DAY22A_AUDIT_CONTRACT_JSON,
    OUT_DAY22A_FORMAT_INVENTORY,
    OUT_DAY22A_TIMEBASE_AUDIT,
    OUT_DAY22A_FILE_INVENTORY,
    OUT_DAY22A_LOAD_SUMMARY,
    OUT_DAY22A_EVENT_AUDIT,
    OUT_DAY22A_Q_SUMMARY,
    OUT_DAY22A_FINALQ_PAIR_AUDIT,
    OUT_DAY22A_RESOLUTION_SUMMARY,
    OUT_DAY22A_SEGMENT_ASSIGNMENT,
    OUT_DAY22A_DTQ_SUMMARY,
    OUT_DAY22A_VERDICT,
]

missing = [p for p in required_day22_closure_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Cannot close Day22A. Missing required files:\n"
        + "\n".join(str(p) for p in missing)
    )

df_day22_verdict = pd.read_csv(OUT_DAY22A_VERDICT)
df_day22_segment = pd.read_csv(OUT_DAY22A_SEGMENT_ASSIGNMENT)
df_day22_dtq = pd.read_csv(OUT_DAY22A_DTQ_SUMMARY)
df_day22_resolution = pd.read_csv(OUT_DAY22A_RESOLUTION_SUMMARY)
df_day22_finalq = pd.read_csv(OUT_DAY22A_FINALQ_PAIR_AUDIT)

closure_cols = [
    "protocol_pair",
    "evidence_status",
    "mechanism_verdict",
    "interpretation_class",
    "Q80_common_segment",
    "Q90_common_segment",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "segment_A_above_floor_status",
    "segment_A_dt_resid_p95_abs_s",
    "segment_A_dt_resid_fit_p95_abs_s",
    "audit_resolution_status",
    "fitted_resolution_status",
    "caveat",
]

df_day22_closure = df_day22_verdict[closure_cols].copy()
df_day22_closure.to_csv(OUT_DAY22A_CLOSURE_CSV, index=False)

print(f"[OK] Wrote Day22A closure CSV: {OUT_DAY22A_CLOSURE_CSV}")
print(f"[OK] closure rows = {df_day22_closure.shape[0]}")
print(df_day22_closure.to_string(index=False))

[OK] Wrote Day22A closure CSV: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step7_closure_summary.csv
[OK] closure rows = 3
              protocol_pair                        evidence_status                                                   mechanism_verdict                                           interpretation_class          Q80_common_segment                     Q90_common_segment  dt_Q80_common_raw_s  dt_Q90_common_raw_s                    segment_A_above_floor_status  segment_A_dt_resid_p95_abs_s  segment_A_dt_resid_fit_p95_abs_s                          audit_resolution_status                              fitted_resolution_status                                                                                                                                                                                                                                                                                                                                      caveat
0.3C DC vs 0.3C

In [17]:
# Day22A Cell 9B — closure Markdown note
#
# Purpose:
# - Write human-readable Day22A closure note
# - No recomputation
# - No new mechanism claim

now_utc = datetime.now(timezone.utc).isoformat()

closure_table = df_day22_closure.to_string(index=False)

resolution_table = df_day22_resolution.to_string(index=False)

segment_table = df_day22_segment[[
    "protocol_pair",
    "Q_Vmax_DC_Ah",
    "Q_Vmax_DCAC_Ah",
    "Q_Vmax_shift_Ah",
    "Q80_common_segment",
    "Q90_common_segment",
    "Q80_nominal_segment",
    "Q90_nominal_segment",
]].to_string(index=False)

finalq_table = df_day22_finalq[[
    "protocol_pair",
    "Q_final_DC_Ah",
    "Q_final_DCAC_Ah",
    "Q_final_diff_mAh",
    "Q_final_diff_status",
]].to_string(index=False)

lines = []

lines.append("# Day22A Closure Note — MJ1 Low-Amplitude Full-Protocol Audit")
lines.append("")
lines.append(f"Generated: `{now_utc}`")
lines.append(f"Git HEAD: `{GIT_HEAD_DAY22A}`")
lines.append(f"Notebook: `{DAY22A_NOTEBOOK_NAME}`")
lines.append("")
lines.append("## 1. Scope")
lines.append("")
lines.append("Day22A applies the Day21A full-protocol segmentation audit to a low-amplitude MJ1 group:")
lines.append("")
lines.append("- `0.3C DC`")
lines.append("- `0.3C + 0.4C 0.1τ`")
lines.append("- `0.3C + 0.4C 1τ`")
lines.append("- `0.3C + 0.4C 10τ`")
lines.append("")
lines.append("The purpose is to test whether the boundary/control-state mediated first-passage gains observed in the `0.3C + 0.7C` group weaken, remain above audit resolution, or become unresolved at lower AC amplitude.")
lines.append("")
lines.append("Day22A does not claim strict disappearance because no independent repeat-based experimental noise floor is available.")
lines.append("")
lines.append("## 2. Data format and timebase")
lines.append("")
lines.append("Day22A includes mixed CSV formats:")
lines.append("")
lines.append("- `0.1τ` and `10τ`: processed 1 Hz aligned CSV without NGU201 metadata")
lines.append("- `1τ` and `0.3C DC`: NGU201 LOG raw format")
lines.append("")
lines.append("All files passed timebase audit with monotonic parsed or unwrapped timestamps.")
lines.append("")
lines.append("## 3. Final-Q consistency")
lines.append("")
lines.append("```text")
lines.append(finalq_table)
lines.append("```")
lines.append("")
lines.append("All Day22A pairs are `final_Q_consistent`. No asymmetric final-Q caveat is required.")
lines.append("")
lines.append("## 4. Experimental audit-resolution estimate")
lines.append("")
lines.append("Day22A uses a DC self-consistency lower-bound floor, obtained from the `0.3C DC` reference by even/odd row splitting.")
lines.append("")
lines.append("```text")
lines.append(resolution_table)
lines.append("```")
lines.append("")
lines.append("This is a lower-bound audit-resolution estimate for sampling, interpolation, and first-passage sensitivity. It is not a repeat-based experimental noise floor.")
lines.append("")
lines.append("## 5. Segment assignment")
lines.append("")
lines.append("```text")
lines.append(segment_table)
lines.append("```")
lines.append("")
lines.append("The key Day22A structural result is:")
lines.append("")
lines.append("- `Q80_common` lies in Segment A for all 0.3C+0.4C protocols.")
lines.append("- `Q90_common` lies in Segment B for all 0.3C+0.4C protocols.")
lines.append("")
lines.append("This differs from Day21A `0.3C+0.7C`, where `1τ` and `10τ` had both Q80/Q90 common anchors in Segment B.")
lines.append("")
lines.append("## 6. Formal verdict")
lines.append("")
lines.append("```text")
lines.append(closure_table)
lines.append("```")
lines.append("")
lines.append("Day22A formal verdicts must be interpreted with low-amplitude audit-resolution caveats.")
lines.append("")
lines.append("For `1τ` and `10τ`, the formal prescribed-geometry audit reopens Segment-A residual because Q80_common lies in Segment A and prescribed residual p95 exceeds the audit floor. However, fitted-waveform diagnostics reduce the p95 residual below the Day22A self-consistency resolution, indicating waveform-geometry mismatch / first-passage sensitivity rather than a confirmed non-geometric electrochemical mechanism.")
lines.append("")
lines.append("For `0.1τ`, the formal residual is intermediate and the fitted waveform diagnostic is unreliable. It is not mechanism evidence.")
lines.append("")
lines.append("## 7. Comparison with Day21A")
lines.append("")
lines.append("Compared with the Day21A `0.3C+0.7C` group, Day22A shows:")
lines.append("")
lines.append("- smaller full-protocol first-passage gains at Q80/Q90 common anchors")
lines.append("- weaker voltage-boundary shift")
lines.append("- Q80_common moving from Segment B to Segment A")
lines.append("- no strict evidence for disappearance")
lines.append("- no confirmed non-geometric Segment-A acceleration mechanism")
lines.append("")
lines.append("The low-amplitude result supports amplitude sensitivity of the boundary/control-state gain pathway, but within the current audit it cannot prove disappearance of the effect.")
lines.append("")
lines.append("## 8. Allowed claims")
lines.append("")
lines.append("Allowed:")
lines.append("")
lines.append("1. Lower AC amplitude reduces Q80/Q90 common first-passage gains relative to Day21A.")
lines.append("2. In Day22A, Q80_common remains in Segment A, while Q90_common lies in Segment B.")
lines.append("3. Day22A formal Segment-A residual is above-floor for 1τ and 10τ under prescribed geometry.")
lines.append("4. Fitted-waveform diagnostics collapse the p95 residual below self-consistency resolution for 1τ and 10τ.")
lines.append("5. Day22A does not support a confirmed non-geometric Segment-A mechanism.")
lines.append("6. The effect cannot be said to disappear without repeat-based noise-floor evidence.")
lines.append("")
lines.append("## 9. Prohibited claims")
lines.append("")
lines.append("Do not claim:")
lines.append("")
lines.append("1. Low-amplitude DC–AC effect disappears.")
lines.append("2. Day22A proves non-geometric Segment-A acceleration.")
lines.append("3. Fitted-waveform residual replaces the formal prescribed-geometry residual.")
lines.append("4. PyBaMM numerical floor is applicable as MJ1 experimental noise floor.")
lines.append("5. Small residuals prove persistence of a mechanism.")
lines.append("")
lines.append("## 10. Key output files")
lines.append("")
lines.append(f"- Raw CSV format inventory: `{OUT_DAY22A_FORMAT_INVENTORY}`")
lines.append(f"- Timebase audit: `{OUT_DAY22A_TIMEBASE_AUDIT}`")
lines.append(f"- File inventory: `{OUT_DAY22A_FILE_INVENTORY}`")
lines.append(f"- Load sanity: `{OUT_DAY22A_LOAD_SUMMARY}`")
lines.append(f"- Event audit: `{OUT_DAY22A_EVENT_AUDIT}`")
lines.append(f"- Q integration summary: `{OUT_DAY22A_Q_SUMMARY}`")
lines.append(f"- Final-Q pair audit: `{OUT_DAY22A_FINALQ_PAIR_AUDIT}`")
lines.append(f"- Resolution floor summary: `{OUT_DAY22A_RESOLUTION_SUMMARY}`")
lines.append(f"- Segment assignment: `{OUT_DAY22A_SEGMENT_ASSIGNMENT}`")
lines.append(f"- Δt summary: `{OUT_DAY22A_DTQ_SUMMARY}`")
lines.append(f"- Verdict: `{OUT_DAY22A_VERDICT}`")
lines.append("")
lines.append("## 11. Closure status")
lines.append("")
lines.append("Day22A is closed as a low-amplitude audit.")
lines.append("")
lines.append("Next recommended step:")
lines.append("")
lines.append("Commit Day22A notebook and audit outputs, excluding raw CSV files.")

closure_md = "\n".join(lines)

OUT_DAY22A_CLOSURE_MD.write_text(closure_md, encoding="utf-8")

print(f"[OK] Wrote Day22A closure note: {OUT_DAY22A_CLOSURE_MD}")
print("[OK] Day22A audit closed.")

[OK] Wrote Day22A closure note: /Users/louislu/pybamm-dcac-superimposed/data/day22A_step7_closure_note.md
[OK] Day22A audit closed.
